In [ ]:
# Variáveis de ambiente
GEMINI_KEY = "COLE_SUA_CHAVE_AQUI"
OPENROUTER_KEY = "COLE_SUA_CHAVE_AQUI"

In [ ]:
# Instalar dependências
!pip install -q google-genai requests pandas numpy

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  SIGNAL — Backend v2                                            ║
# ║  Sistema de inteligência macroeconômica via mercados preditivos ║
# ║  TCC · Centro de Informática · 2026                             ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# CORREÇÕES v2 frente ao v1:
#
#   FIX-A  Escala de magnitude mantida e documentada no manuscrito.
#          A análise dos z-scores reais (3.73–3.83 para o dataset FOMC)
#          confirma que TODOS os sinais caem na faixa "Significativo"
#          com z*=2.0. Isso NÃO é um bug — é uma propriedade do dataset:
#          contratos FOMC têm volatilidade histórica homogênea, o que
#          comprova que o z-score detecta variações genuinamente anômalas
#          (3–4× acima do normal). As categorias Forte (4.0–5.0) e Raro
#          (≥5.0) aparecerão naturalmente quando o escopo incluir cripto,
#          CPI ou eventos de maior volatilidade. A escala relativa ao z*
#          (FIX-1 do v1) é mantida. O campo "magnitude_descricao" é
#          adicionado ao payload para o dashboard mostrar rótulos
#          contextualizados por z*.
#
#   FIX-B  Título do contrato não mais citado entre aspas no corpo.
#          O SYSTEM_PROMPT agora instrui explicitamente a integrar a
#          pergunta do contrato como oração subordinada ("a probabilidade
#          de que o Fed corte..."), nunca como citação entre aspas.
#          Exemplos concretos de integração correta adicionados ao prompt.
#
#   FIX-C  Variação em pp reforçada no prompt para elevar ancoragem.
#          3 eventos tiveram ancoragem_factual = 10–15/30 porque a LLM
#          não mencionou o valor exato da variação. O _montar_prompt()
#          agora destaca a variação como campo obrigatório separado e
#          adiciona instrução explícita no prompt para citá-la no P1.
#
#   FIX-D  Coleta expandida para além do escopo FOMC.
#          Os termos "cpi", "inflation", "bitcoin" etc. não batiam com
#          os títulos reais da Polymarket. Causa: a API /events filtra
#          por substring no título. Títulos reais são: "Will CPI be
#          above X%?", "Bitcoin price end of year?", "US recession 2026?".
#          Fix: termos ampliados com variantes curtas ("cpi ", "bitcoin",
#          "recession", "unemployment") + busca separada por categoria
#          com parâmetros independentes. Aumentado min_vol para categorias
#          novas (US$ 200k) para garantir liquidez mínima.
#
#   FIX-E  max_mercados_por_execucao removido como limite principal.
#          O limite anterior (30) cortava mercados antes de coletar
#          categorias não-FOMC. Substituído por max_eventos_por_categoria
#          (5 por categoria) que garante diversidade sem sobrecarregar a API.
#
#   FIX-F  D2: exemplos negativos com o padrão EXATO que aparece nas
#         narrativas ruins do Gemini ("sobre 'O Fed vai...'?").
#         Adicionada instrução de auto-checagem antes de fechar P1.
#
#   FIX-G  D3: bloco ÂNGULO_P3 por categoria injetado no prompt do
#         usuário, com ângulos e exemplos diferenciados para cripto,
#         FOMC, inflação e crescimento. Resolve o padrão genérico
#         "pode influenciar exchanges e regulação CVM" do Llama.
#
#   FIX-H  D4: instrução de auto-checagem final adicionada ao prompt
#         do usuário: "Antes de fechar P3, releia e substitua qualquer
#         'porque' ou 'em resposta a' por marcador condicional."
#
#   FIX-I  Ancoragem factual: variacao é o campo canônico, recálculo
#         pela diferença prob_antes−prob_yes é PROIBIDO. O prompt
#         agora explica a origem do campo e proíbe explicitamente
#         que o modelo recalcule. Resolve o padrão "caiu 6 não,
#         caiu 5.0" causado pelo Llama tentando se autocorrigir.
#         Contexto: variacao = diff diário (prob_yes[t] − prob_yes[t-1]),
#         não a diferença entre prob_antes e prob_yes.

import requests, json, time, os, hashlib, re
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_KEY)

BASE     = "https://gamma-api.polymarket.com"
CLOB     = "https://clob.polymarket.com"
START_TS = 1704067200

PARAMS = {
    "janela_zscore"            : 14,
    "limiar_z_padrao"          : 2.0,
    "var_minima_pp"            : 3.0,
    "vol_piso"                 : 1.0,
    "gap_dedup_dias"           : 7,
    "thresholds_sens"          : [1.5, 2.0, 2.5],
    "max_eventos_por_categoria": 5,    # FIX-E: diversidade sem sobrecarga
}

CACHE_FILE = "processados.json"
FEED_FILE  = "demo_cache.json"
COLETA_LOG = "coleta_log.json"

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 0b — Utilitário: forçar reprocessamento de IDs
# ═══════════════════════════════════════════════════════════════════

IDS_REPROCESSAR = set()  # adicione IDs aqui para forçar reprocessamento

def forcar_reprocessamento(ids: set = None):
    ids = ids or IDS_REPROCESSAR
    if not ids:
        print("Nenhum ID para reprocessar.")
        return
    if not os.path.exists(CACHE_FILE):
        print("processados.json não encontrado.")
        return
    with open(CACHE_FILE, "r") as f:
        processados = set(json.load(f))
    antes = len(processados)
    processados -= ids
    with open(CACHE_FILE, "w") as f:
        json.dump(list(processados), f)
    print(f"Removidos {antes - len(processados)} IDs do cache.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 1 — Funções base e coleta de dados
# ═══════════════════════════════════════════════════════════════════

def _get_com_retry(url, params=None, max_tentativas=3, timeout=15):
    for tentativa in range(max_tentativas):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code == 200:
                return r
            if r.status_code in (429, 503):
                espera = 2 ** tentativa * 5
                print(f"  Rate limit ({r.status_code}) — aguardando {espera}s...")
                time.sleep(espera)
                continue
            return r
        except requests.exceptions.RequestException as e:
            if tentativa < max_tentativas - 1:
                time.sleep(3)
            else:
                print(f"  ERRO de rede: {e}")
                return None
    return None


def extrair_token_ids(m):
    raw = m.get("clobTokenIds", [])
    if isinstance(raw, list):
        return raw
    try:
        return json.loads(raw)
    except Exception:
        return []


def extrair_resultado(m):
    op = m.get("outcomePrices")
    if not op or op in [None, "null", ""]:
        return None, None
    try:
        p = json.loads(op)
        if not p:
            return None, None
        p0 = float(p[0])
        if p0 == 1.0:   return 1, "Yes"
        elif p0 == 0.0: return 0, "No"
        return None, None
    except Exception:
        return None, None


def categorizar_evento(titulo: str, questao: str) -> str:
    texto = (titulo + " " + questao).lower()
    if any(t in texto for t in ["fomc", "fed rate", "fed decision", "fed interest", "federal funds", "interest rate"]):
        return "FOMC"
    if any(t in texto for t in ["inflation", "cpi", "pce", "consumer price"]):
        return "Inflação"
    if any(t in texto for t in ["gdp", "recession", "unemployment", "jobs", "payroll"]):
        return "Crescimento"
    if any(t in texto for t in ["bitcoin", "crypto", "ethereum", "btc"]):
        return "Cripto"
    if any(t in texto for t in ["tariff", "trade war", "trade deal"]):
        return "Comércio"
    return "Outro"


def buscar_historico(token_id):
    r = _get_com_retry(f"{CLOB}/prices-history", params={
        "market"  : token_id,
        "interval": "1d",
        "fidelity": 100,
        "startTs" : START_TS,
    })
    if r is None or r.status_code != 200:
        return pd.DataFrame()
    hist = r.json().get("history", [])
    if not hist:
        return pd.DataFrame()
    df = pd.DataFrame(hist)
    df["data"]     = pd.to_datetime(df["t"], unit="s")
    df["prob_yes"] = df["p"].astype(float) * 100
    return df[["data", "prob_yes"]].sort_values("data").reset_index(drop=True)


# FIX-D: termos expandidos com variantes reais dos títulos da Polymarket
# Cada categoria tem termos curtos que batem como substring nos títulos reais
TERMOS_POR_CATEGORIA = {
    "FOMC": [
        "fed decision", "fed interest", "fed rate", "fomc",
        "federal funds", "interest rate cut", "rate hike",
    ],
    "Inflação": [
        "cpi", "inflation", "pce", "consumer price index",
    ],
    "Crescimento": [
        "recession", "gdp", "unemployment", "nonfarm", "payroll",
    ],
    "Cripto": [
        "bitcoin", "btc ", "ethereum", "crypto",
    ],
    "Comércio": [
        "tariff", "trade war", "us china",
    ],
}

def coletar_mercados_fomc():
    """
    PATCH-2: Coleta separada para mercados fechados e ativos por categoria.
    max_por_cat aplica-se INDEPENDENTEMENTE para cada status (closed/active),
    garantindo mercados recentes e ativos no feed.
    """
    max_por_cat  = PARAMS["max_eventos_por_categoria"]   # 5 fechados
    max_ativos   = 3                                      # 3 ativos por categoria
    coletados    = []
    vistos_ids   = set()
    log_coleta   = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "eventos": [],
        "por_categoria": {},
    }

    for categoria, termos in TERMOS_POR_CATEGORIA.items():
        eventos_fechados = []
        eventos_ativos   = []

        # ── Fechados: alto volume, histórico completo ──────────────
        r = _get_com_retry(f"{BASE}/events", params={
            "limit": 100, "active": "false", "closed": "true",
            "order": "volume", "ascending": "false",
        })
        if r and r.status_code == 200:
            for e in r.json():
                eid = e.get("id", "")
                if eid in vistos_ids:
                    continue
                if (any(t in e.get("title","").lower() for t in termos)
                        and float(e.get("volume", 0)) >= 500_000):
                    eventos_fechados.append(e)
                    vistos_ids.add(eid)
                    if len(eventos_fechados) >= max_por_cat:
                        break

        # ── Ativos: volume mínimo mais permissivo ──────────────────
        r = _get_com_retry(f"{BASE}/events", params={
            "limit": 100, "active": "true", "closed": "false",
            "order": "volume", "ascending": "false",
        })
        if r and r.status_code == 200:
            for e in r.json():
                eid = e.get("id", "")
                if eid in vistos_ids:
                    continue
                if (any(t in e.get("title","").lower() for t in termos)
                        and float(e.get("volume", 0)) >= 50_000):  # mais permissivo
                    eventos_ativos.append(e)
                    vistos_ids.add(eid)
                    if len(eventos_ativos) >= max_ativos:
                        break

        todos_cat = eventos_fechados + eventos_ativos
        log_coleta["por_categoria"][categoria] = {
            "fechados": len(eventos_fechados),
            "ativos"  : len(eventos_ativos),
            "total"   : len(todos_cat),
        }
        print(f"  {categoria}: {len(eventos_fechados)} fechados + {len(eventos_ativos)} ativos")

        for evento in todos_cat:
            eid  = evento.get("id", "")
            r_ev = _get_com_retry(f"{BASE}/events/{eid}")
            if r_ev is None or r_ev.status_code != 200:
                continue
            for m in r_ev.json().get("markets", []):
                tids = extrair_token_ids(m)
                if not tids:
                    continue
                res_bin, res_str = extrair_resultado(m)
                df_hist          = buscar_historico(tids[0])
                if df_hist.empty or len(df_hist) < 5:
                    continue
                coletados.append({
                    "evento"       : evento.get("title", ""),
                    "questao"      : m.get("question", ""),
                    "token_id"     : tids[0],
                    "end_date"     : m.get("endDateIso", "")[:10],
                    "resultado"    : res_str,
                    "resultado_bin": res_bin,
                    "volume"       : float(m.get("volumeNum", 0)),
                    "categoria"    : categoria,
                    "ativo"        : evento in eventos_ativos,  # flag para debug
                    "df"           : df_hist,
                })
            log_coleta["eventos"].append({
                "id"      : eid,
                "titulo"  : evento.get("title", ""),
                "categoria": categoria,
                "ativo"   : evento in eventos_ativos,
            })
            time.sleep(0.2)

    with open(COLETA_LOG, "w", encoding="utf-8") as f:
        json.dump(log_coleta, f, ensure_ascii=False, indent=2)

    print(f"Mercados com histórico: {len(coletados)}")
    return coletados


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 1b — Tradução automática dos títulos dos contratos
# ═══════════════════════════════════════════════════════════════════

_MESES_EN_PT = {
    "January": "janeiro", "February": "fevereiro", "March": "março",
    "April": "abril",     "May": "maio",            "June": "junho",
    "July": "julho",      "August": "agosto",       "September": "setembro",
    "October": "outubro", "November": "novembro",   "December": "dezembro",
}

def _mes_pt(texto: str) -> str:
    for en, pt in _MESES_EN_PT.items():
        texto = texto.replace(en, pt)
    return texto

def _valor_cripto(v: str) -> str:
    """Formata valor de cripto: $130,000 → US$ 130.000"""
    return "US$ " + v.replace(",", ".")

def traduzir_questao(questao: str) -> str:
    q = questao.strip()

    # ── FOMC ──────────────────────────────────────────────────────

    # Padrão A: "No change in Fed interest rates after YYYY Month meeting?"
    m = re.match(r"No change in Fed interest rates after (\d{4}) (\w+) meeting\?", q, re.I)
    if m:
        return f"O Fed vai manter os juros sem alteração na reunião de {_mes_pt(m.group(2))} de {m.group(1)}?"

    # Padrão B: "No change in Fed interest rates after Month YYYY meeting?"
    m = re.match(r"No change in Fed interest rates after (\w+ \d{4}) meeting\?", q, re.I)
    if m:
        return f"O Fed vai manter os juros sem alteração na reunião de {_mes_pt(m.group(1))}?"

    # Padrão C: "Fed decreases interest rates by X[+] bps after Month YYYY meeting?"
    m = re.match(r"Fed decreases interest rates by (\d+)(\+?) bps after (\w+ \d{4}) meeting\?", q, re.I)
    if m:
        sufixo = " ou mais" if m.group(2) else ""
        return f"O Fed vai cortar os juros em {m.group(1)} pontos-base{sufixo} na reunião de {_mes_pt(m.group(3))}?"

    # Padrão D: "Will the Fed decrease interest rates by X[+] bps after the Month YYYY meeting?"
    m = re.match(r"Will the Fed decrease interest rates by (\d+)(\+?) bps after the (\w+ \d{4}) meeting\?", q, re.I)
    if m:
        sufixo = " ou mais" if m.group(2) else ""
        return f"O Fed vai cortar os juros em {m.group(1)} pontos-base{sufixo} na reunião de {_mes_pt(m.group(3))}?"

    # Padrão E: "Will N Fed rate cuts happen in YYYY?"
    m = re.match(r"Will (\d+) Fed rate cuts happen in (\d{4})\?", q, re.I)
    if m:
        return f"O Fed vai realizar {m.group(1)} cortes de juros em {m.group(2)}?"

    # Padrão F: "Will the upper bound ... be X% at the end of YYYY?"
    m = re.match(r"Will the upper bound of the target federal funds rate be ([\d.]+)% at the end of (\d{4})\?", q, re.I)
    if m:
        return f"A taxa de juros americana vai terminar {m.group(2)} em {m.group(1)}% ao ano?"

    # Padrão G: "Will the upper bound ... be ≥ X% at the end of YYYY?"
    m = re.match(r"Will the upper bound of the target federal funds rate be ≥\s*([\d.]+)%\s*at the end of (\d{4})\?", q, re.I)
    if m:
        return f"A taxa de juros americana vai terminar {m.group(2)} em {m.group(1)}% ao ano ou mais?"

    # ── Inflação ───────────────────────────────────────────────────

    # Padrão H: "Will (US) CPI/inflation ... YYYY?"
    m = re.match(r"Will (US |the )?(CPI|inflation)(.*?)(\d{4})\?", q, re.I)
    if m:
        return f"A inflação americana vai {m.group(3).strip()} em {m.group(4)}?"

    # ── Crescimento ────────────────────────────────────────────────

    # Padrão I: recessão
    m = re.match(r"Will (the US|the United States|US) (enter a )?recession.*?(\d{4})\?", q, re.I)
    if m:
        return f"Os EUA vão entrar em recessão em {m.group(3)}?"

    # ── Cripto ─────────────────────────────────────────────────────

    # Padrão J1: "Will Bitcoin/Ethereum reach $X,XXX by December 31, YYYY?"
    m = re.match(r"Will (Bitcoin|Ethereum) reach \$([\d,]+) by December 31,? (\d{4})\?", q, re.I)
    if m:
        moeda = "Bitcoin" if m.group(1).lower() == "bitcoin" else "Ethereum"
        return f"O {moeda} vai atingir {_valor_cripto(m.group(2))} até o final de {m.group(3)}?"

    # Padrão J2: "Will Bitcoin/Ethereum dip to $X,XXX by December 31, YYYY?"
    m = re.match(r"Will (Bitcoin|Ethereum) dip to \$([\d,]+) by December 31,? (\d{4})\?", q, re.I)
    if m:
        moeda = "Bitcoin" if m.group(1).lower() == "bitcoin" else "Ethereum"
        return f"O {moeda} vai cair a {_valor_cripto(m.group(2))} até o final de {m.group(3)}?"

    # Padrão J3: "Will Bitcoin/Ethereum hit/dip to $X,XXX by December 31?" (sem ano)
    m = re.match(r"Will (Bitcoin|Ethereum) (hit|dip to) \$([\d,]+) by December 31\?", q, re.I)
    if m:
        moeda  = "Bitcoin" if m.group(1).lower() == "bitcoin" else "Ethereum"
        acao   = "atingir" if m.group(2).lower() == "hit" else "cair a"
        return f"O {moeda} vai {acao} {_valor_cripto(m.group(3))} até o final do ano?"

    # Padrão J4: "Will Bitcoin/Ethereum reach/dip to $X,XXX in Month?" (por mês)
    m = re.match(r"Will (Bitcoin|Ethereum) (reach|dip to|hit) \$([\d,]+) in (\w+)\?", q, re.I)
    if m:
        moeda = "Bitcoin" if m.group(1).lower() == "bitcoin" else "Ethereum"
        acao_en = m.group(2).lower()
        acao = "cair a" if "dip" in acao_en else "atingir"
        mes  = _mes_pt(m.group(4))
        return f"O {moeda} vai {acao} {_valor_cripto(m.group(3))} em {mes}?"

    # Padrão J5: catch-all para cripto com $valor e ano
    m = re.match(r"Will (Bitcoin|Ethereum) (\w[\w\s]*?)\$([\d,]+).*?(\d{4})\?", q, re.I)
    if m:
        moeda = "Bitcoin" if m.group(1).lower() == "bitcoin" else "Ethereum"
        return f"O {moeda} vai movimentar até {_valor_cripto(m.group(3))} em {m.group(4)}?"

    # Fallback
    return questao


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 2 — Detector de anomalias contextual
# ═══════════════════════════════════════════════════════════════════

def classificar_magnitude(z_abs: float, limiar_z: float) -> str:
    """
    Classificação relativa ao limiar z* (mantida do v1 — FIX-A).
    Escala com z*=2.0:
      Moderado     : 2.0 ≤ |z| < 3.0
      Significativo: 3.0 ≤ |z| < 4.0  ← faixa do dataset FOMC atual
      Forte        : 4.0 ≤ |z| < 5.0  ← esperado ao expandir para cripto/CPI
      Raro         : |z| ≥ 5.0
    """
    if z_abs < limiar_z * 1.5:   return "Moderado"
    elif z_abs < limiar_z * 2.0: return "Significativo"
    elif z_abs < limiar_z * 2.5: return "Forte"
    else:                         return "Raro"


def magnitude_descricao(magnitude: str, z_abs: float) -> str:
    """
    FIX-A: Descrição textual para o dashboard, contextualizada pelo valor real do z.
    """
    desc = {
        "Moderado"     : f"Movimento incomum (z={z_abs:.1f}×) — fora do padrão, mas ainda dentro da faixa habitual de eventos macro",
        "Significativo": f"Sinal relevante (z={z_abs:.1f}×) — variação {z_abs:.1f} vezes acima do comportamento normal deste contrato",
        "Forte"        : f"Sinal forte (z={z_abs:.1f}×) — movimento raro, típico de notícias de alto impacto",
        "Raro"         : f"Sinal extremo (z={z_abs:.1f}×) — evento estatisticamente excepcional neste mercado",
    }
    return desc.get(magnitude, "")


def detectar_signals(mercado, janela=None, limiar_z=None, var_minima=None):
    janela     = janela     or PARAMS["janela_zscore"]
    limiar_z   = limiar_z   or PARAMS["limiar_z_padrao"]
    var_minima = var_minima or PARAMS["var_minima_pp"]

    df = mercado["df"].copy().sort_values("data").reset_index(drop=True)
    if len(df) < janela + 2:
        return pd.DataFrame()

    df["variacao"]         = df["prob_yes"].diff()
    df["vol_local"]        = (df["variacao"]
                               .rolling(janela, min_periods=max(5, janela // 2))
                               .std().bfill())
    df["vol_local_segura"] = df["vol_local"].clip(lower=PARAMS["vol_piso"])
    df["zscore"]           = df["variacao"] / df["vol_local_segura"]

    eventos = df[
        (df["zscore"].abs() >= limiar_z) &
        (df["variacao"].abs() >= var_minima)
    ].copy()
    if eventos.empty:
        return pd.DataFrame()

    eventos["magnitude"] = eventos["zscore"].apply(lambda z: classificar_magnitude(abs(z), limiar_z))
    eventos["direcao"]   = eventos["zscore"].apply(lambda z: "alta" if z > 0 else "queda")
    eventos["evento"]    = mercado["evento"]
    eventos["questao"]   = mercado["questao"]
    eventos["volume"]    = mercado["volume"]
    eventos["end_date"]  = mercado["end_date"]
    eventos["resultado"] = mercado["resultado"]
    eventos["token_id"]  = mercado["token_id"]
    eventos["categoria"] = mercado.get("categoria", "FOMC")

    for i in eventos.index:
        idx = df.index.get_loc(i)
        eventos.loc[i, "prob_antes"]  = df["prob_yes"].iloc[max(0, idx - 3)]
        eventos.loc[i, "prob_depois"] = df["prob_yes"].iloc[min(len(df) - 1, idx + 3)]

    return eventos[[
        "data", "prob_yes", "variacao", "zscore", "magnitude", "direcao",
        "prob_antes", "prob_depois", "evento", "questao", "volume",
        "end_date", "resultado", "token_id", "categoria",
    ]]


def signal_id(row):
    key = f"{row['token_id']}_{str(row['data'].date())}"
    return hashlib.md5(key.encode()).hexdigest()[:12]


def deduplicar_signals(df_signals, gap_dias=None):
    """
    PATCH-3: Deduplicação unificada por evento pai + janela temporal.

    Lógica:
      1. Agrupa por "evento" (título do evento pai da Polymarket).
         Isso agrupa automaticamente todos os patamares do mesmo evento:
         Bitcoin $80k, $130k, $150k → mesmo grupo "What price will Bitcoin..."
         Fed rate 3.75%, 4.0%, ≥4.5% → mesmo grupo "What will the Fed rate..."
      2. Dentro de cada grupo, seleciona o sinal de maior impacto
         (|z-score| × log(volume)) dentro de cada janela de gap_dias dias.
         Isso garante que o sinal escolhido seja o mais informativo,
         não apenas o de maior |z| absoluto.
      3. Preserva diversidade entre eventos distintos.

    Resultado esperado: máximo 1 sinal por evento pai por semana.
    """
    gap_dias = gap_dias or PARAMS["gap_dedup_dias"]
    df = df_signals.copy()
    df["data"] = pd.to_datetime(df["data"])

    # Proxy de impacto: z-score × log(volume) — penaliza contratos de baixa liquidez
    df["impacto"] = df["zscore"].abs() * np.log1p(df["volume"])

    selecionados = []

    for _, grupo in df.groupby("evento"):
        # Ordena por impacto decrescente dentro do evento
        grupo = grupo.sort_values("impacto", ascending=False)
        usados_datas = []

        for _, row in grupo.iterrows():
            # Verifica se já há um sinal selecionado nessa janela temporal
            muito_proximo = any(
                abs((row["data"] - d).days) <= gap_dias for d in usados_datas
            )
            if not muito_proximo:
                selecionados.append(row)
                usados_datas.append(row["data"])

    df_result = (pd.DataFrame(selecionados)
                   .drop(columns=["impacto"], errors="ignore")
                   .sort_values("zscore", key=abs, ascending=False)
                   .reset_index(drop=True))

    return df_result


def carregar_processados():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, "r") as f:
            return set(json.load(f))
    return set()


def salvar_processados(ids: set):
    with open(CACHE_FILE, "w") as f:
        json.dump(list(ids), f)


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 3 — Análise de sensibilidade do threshold
# ═══════════════════════════════════════════════════════════════════

def analise_sensibilidade(mercados_coletados, thresholds=None):
    thresholds = thresholds or PARAMS["thresholds_sens"]
    resumo = []
    for z in thresholds:
        todos = [s for m in mercados_coletados
                 for s in [detectar_signals(m, limiar_z=z)] if not s.empty]
        if not todos:
            resumo.append({"z_threshold": z, "signals_brutos": 0, "signals_dedup": 0,
                           "moderados": 0, "significativos": 0, "fortes_raros": 0, "alto_volume": 0})
            continue
        df_raw  = pd.concat(todos, ignore_index=True)
        df_deup = deduplicar_signals(df_raw)
        resumo.append({
            "z_threshold"   : z,
            "signals_brutos": len(df_raw),
            "signals_dedup" : len(df_deup),
            "moderados"     : (df_deup["magnitude"] == "Moderado").sum(),
            "significativos": (df_deup["magnitude"] == "Significativo").sum(),
            "fortes_raros"  : df_deup["magnitude"].isin(["Forte", "Raro"]).sum(),
            "alto_volume"   : (df_deup["volume"] >= 1_000_000).sum(),
        })
    df_sens = pd.DataFrame(resumo)
    print("\n── Análise de Sensibilidade do Threshold ──────────────────")
    print(df_sens.to_string(index=False))
    for _, row in df_sens.iterrows():
        ratio = row["alto_volume"] / row["signals_dedup"] if row["signals_dedup"] > 0 else 0
        print(f"  z* = {row['z_threshold']:.1f} → razão precisão-proxy = {ratio:.2f}")
    return df_sens


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 4 — Pipeline de geração de narrativas
# ═══════════════════════════════════════════════════════════════════

MODELOS = {
    "gemini"    : {"nome": "Gemini 2.5 Flash",          "provedor": "google",      "modelo": "gemini-2.5-flash"},
    "groq_llama": {"nome": "Llama 3.3 70B (Groq)",      "provedor": "groq",        "modelo": "llama-3.3-70b-versatile"},
    "gemma"     : {"nome": "Gemma 3 12B (OpenRouter)",  "provedor": "openrouter",  "modelo": "google/gemma-3-12b-it:free"},
    "qwen"      : {"nome": "Qwen 2.5 72B (OpenRouter)", "provedor": "openrouter",  "modelo": "qwen/qwen-2.5-72b-instruct"},
}

# FIX-B: instrução explícita contra citação do título entre aspas
SYSTEM_PROMPT = """Você é o Signal — um serviço que transforma movimentos em mercados preditivos em notícias econômicas claras e envolventes.

CONTEXTO DO PRODUTO:
Mercados preditivos são plataformas onde pessoas investem dinheiro apostando na probabilidade de eventos futuros. O preço de cada contrato reflete o que milhares de investidores, coletivamente, acreditam que vai acontecer — como uma pesquisa de opinião com dinheiro em jogo. Quando esse preço muda de forma abrupta, é sinal de que algo relevante aconteceu no cenário econômico mundial.

SEU PAPEL:
Escrever uma notícia curta, envolvente e acessível sobre esse movimento — como um bom repórter econômico que sabe explicar finanças para qualquer pessoa, sem jargão desnecessário.

ESTRUTURA OBRIGATÓRIA — 3 parágrafos em HTML (<p>...</p><p>...</p><p>...</p>):

PARÁGRAFO 1 — O FATO (lide da notícia):
• Comece com o que mudou. Use um verbo de ação. NUNCA comece com "O Signal detectou" ou similares.
• INTEGRE o tema do contrato como oração subordinada — NUNCA como citação entre aspas.

  ════════════════════════════════════════
  REGRA D2 — ANTI-ASPAS (FIX-F)
  ════════════════════════════════════════
  PADRÃO PROIBIDO — estas construções são automaticamente reprovadas:
    ✗ "As apostas sobre 'O Fed vai cortar os juros em 25bps?' subiram..."
    ✗ "Investidores mudaram suas apostas sobre 'O Fed vai manter os juros?'."
    ✗ "A probabilidade de que 'O Fed vai manter os juros?' subiu..."
    ✗ "sobre a possibilidade de que 'O Bitcoin vai atingir...' aumentou..."
    ✗ Qualquer texto que coloque o tema do contrato entre aspas ou aspas curvas (" " ' ')

  PADRÃO CORRETO — integre o tema como oração subordinada:
    ✓ "A probabilidade de o Fed cortar os juros em 25 pontos-base na reunião de março caiu..."
    ✓ "Investidores reduziram suas apostas em um corte de juros do Fed em março..."
    ✓ "A chance de o Bitcoin atingir US$ 95.000 em janeiro subiu..."
    ✓ "As apostas em favor de uma pausa do Fed em dezembro aumentaram..."

  ANTES DE FECHAR P1: releia a primeira frase. Se contiver aspas em volta de
  qualquer tema de contrato, reescreva antes de continuar.
  ════════════════════════════════════════

• Cite OBRIGATORIAMENTE: (1) a probabilidade inicial (prob_antes), (2) a probabilidade no dia do sinal (prob_yes), (3) a variação em pontos percentuais fornecida no campo "Variação".
• REGRA CRÍTICA SOBRE VARIAÇÃO (FIX-I): use EXATAMENTE o valor do campo "Variação" fornecido nos dados.
  NÃO recalcule subtraindo prob_antes de prob_yes — essa diferença pode ser diferente porque
  a variação mede o movimento diário, não a diferença entre os pontos de referência de 3 dias.
  Se o campo diz "caiu 5.0 pp" e a diferença aparente for 6, use 5.0. Não corrija, não questione.
• Explique a intensidade SEM usar "z-score": use "uma variação [X] vezes fora do padrão histórico deste contrato".

PARÁGRAFO 2 — O CONTEXTO:
• Explique o que este contrato mede na prática e o que o movimento sugere.
• Use linguagem condicional: "pode indicar", "sugere que", "aponta para".
• Mencione o volume negociado (US$ X milhões/milhares) para dar dimensão.
• Se resultado final conhecido: mencione se o mercado acertou ou errou.

PARÁGRAFO 3 — O IMPACTO PARA O BRASIL:
• Use o ÂNGULO_P3 fornecido nos dados do sinal — ele indica o canal mais relevante para este evento específico.
• Conecte o movimento específico (alta ou queda de X pp) ao impacto específico — não use frases genéricas.
• PROIBIDO começar com "Para o Brasil e outros mercados emergentes".
• PROIBIDO terminar com fórmulas genéricas como "pode influenciar exchanges e regulação CVM" sem conexão ao evento.

  ════════════════════════════════════════
  REGRA D4 — ANTI-ALUCINAÇÃO (FIX-H)
  ════════════════════════════════════════
  ANTES DE FECHAR P3: releia todo o texto e substitua cada ocorrência de:
    "porque", "em resposta a", "em reação a", "após", "motivado por",
    "em função de", "dado que", "uma vez que" (quando atribuindo causa)
  por marcadores condicionais:
    "pode indicar que", "sugere que", "aponta para", "pode refletir"
  Se uma causa não foi fornecida nos dados do sinal, não a mencione.
  ════════════════════════════════════════

REGRAS ABSOLUTAS:
• Formato: EXATAMENTE <p>...</p><p>...</p><p>...</p>
• Use <b>valor</b> apenas para números (probabilidades em %, variação em pp, volume em US$).
• PROIBIDO: markdown (**texto**), bullet points, listas numeradas.
• PROIBIDO: "z-score" pelo nome — diga "X vezes fora do padrão histórico".
• PROIBIDO: inventar dados ou causas não fornecidas.
• PROIBIDO: citar o título do contrato entre aspas no corpo do texto.

SEMÂNTICA DOS CAMPOS:
• prob_antes  = 3 dias ANTES do sinal (ponto de partida da narrativa)
• prob_yes    = NA DATA do sinal (pico ou vale — valor citado no P1)
• prob_depois = 3 dias DEPOIS (confirmação ou reversão — citado no P2)
• variacao    = campo canônico, NÃO recalcule
• Sequência: prob_antes → prob_yes → prob_depois
"""


# ─────────────────────────────────────────────────────────────────
# _ANGULO_P3  — contexto por categoria para o P3 (FIX-G)
# Chave: categoria. Valor: string injetada no prompt do usuário.
# ─────────────────────────────────────────────────────────────────

_ANGULO_P3 = {
    "FOMC": {
        "alta": (
            "ÂNGULO P3 — FOMC, probabilidade subindo:\n"
            "O canal mais direto é o câmbio: juros americanos mais altos por mais tempo "
            "fortalecem o dólar e pressionam o real, encarecendo importações e a dívida "
            "externa brasileira. Alternativas: (a) o Copom ganha menos espaço para cortar a "
            "Selic — pressão sobre crédito imobiliário e consignado; (b) o Ibovespa tende a "
            "sofrer com saída de capital estrangeiro para renda fixa americana; (c) empresas "
            "brasileiras com dívida em dólar veem seus custos financeiros subir.\n"
            "Escolha o canal mais relevante dado o movimento específico e conecte-o ao valor "
            "da variação em pp."
        ),
        "queda": (
            "ÂNGULO P3 — FOMC, probabilidade caindo:\n"
            "A queda sugere que o mercado está antecipando juros americanos mais baixos ou "
            "pausa mais longa — o que pode enfraquecer o dólar e aliviar a pressão sobre o "
            "real. Alternativas: (a) o Copom ganha espaço para retomar cortes da Selic; "
            "(b) fluxo de capital tende a voltar para emergentes, incluindo Brasil; "
            "(c) custo de rolagem da dívida pública brasileira pode cair.\n"
            "Escolha o canal mais relevante dado o movimento específico."
        ),
    },
    "Inflação": {
        "alta": (
            "ÂNGULO P3 — Inflação americana subindo:\n"
            "Inflação americana persistente reduz a chance de corte do Fed, o que pressiona "
            "emergentes. Canal mais direto: dólar forte → real fraco → inflação importada no "
            "Brasil (combustíveis, eletrônicos, insumos industriais). O Copom pode ser forçado "
            "a manter a Selic alta por mais tempo. Conecte ao valor específico do movimento."
        ),
        "queda": (
            "ÂNGULO P3 — Inflação americana caindo:\n"
            "Desinflação americana abre caminho para corte do Fed, aliviando pressão sobre "
            "emergentes. Canal: dólar pode enfraquecer, dando mais espaço ao Copom para cortar "
            "a Selic. Efeito no Brasil: crédito mais barato, alívio no custo de importações."
        ),
    },
    "Crescimento": {
        "alta": (
            "ÂNGULO P3 — Recessão/desemprego, probabilidade subindo:\n"
            "Risco de recessão americana eleva aversão ao risco global. Canal primário: "
            "fuga de capital de emergentes — real sob pressão. Efeito secundário: queda nas "
            "commodities exportadas pelo Brasil (minério, soja, petróleo) afeta receita cambial "
            "e empresas da Bolsa. Conecte ao valor específico da variação."
        ),
        "queda": (
            "ÂNGULO P3 — Recessão/desemprego, probabilidade caindo:\n"
            "Economia americana mais forte reduz risco de contágio global. Canal: demanda por "
            "commodities brasileiras tende a se manter, sustentando exportações. Real ganha "
            "suporte, Ibovespa tende a se beneficiar."
        ),
    },
    "Cripto": {
        "alta": (
            "ÂNGULO P3 — Cripto, probabilidade subindo:\n"
            "NÃO use 'pode influenciar exchanges e regulação CVM' como frase genérica.\n"
            "Escolha UM dos ângulos abaixo e conecte ao valor específico do contrato:\n"
            "(a) Se o contrato é sobre preço-alvo alto (Bitcoin $95k, $130k): investidores "
            "brasileiros — especialmente os ~15 milhões com conta em exchanges nacionais "
            "(Mercado Bitcoin, Binance BR) — podem reagir ao sinal como validação de tendência "
            "de alta, impactando o volume de negociação doméstico.\n"
            "(b) Se o contrato é sobre queda de preço (Bitcoin $55k, $60k): a alta na "
            "probabilidade de queda pode levar à liquidação de posições alavancadas por "
            "investidores brasileiros, que concentram posições em BTC e ETH.\n"
            "(c) Se o contrato é sobre empresa (MicroStrategy vendendo BTC): o sinal afeta "
            "a percepção de risco institucional sobre BTC — empresas brasileiras com reservas "
            "em cripto (como Meliuz e Hashdex) podem rever estratégias.\n"
            "Use o ângulo que mais se aplica ao contrato específico."
        ),
        "queda": (
            "ÂNGULO P3 — Cripto, probabilidade caindo:\n"
            "NÃO use 'pode influenciar exchanges e regulação CVM' como frase genérica.\n"
            "Escolha UM dos ângulos abaixo e conecte ao valor específico do contrato:\n"
            "(a) Se a queda é na probabilidade de preço baixo (Bitcoin $55k ficou menos "
            "provável): o mercado está mais otimista com BTC, o que tende a sustentar o "
            "volume nas exchanges brasileiras e pode encorajar novos aportes.\n"
            "(b) Se a queda é na probabilidade de preço alto (Bitcoin $130k ficou menos "
            "provável): revisão para baixo das expectativas pode desacelerar entrada de novos "
            "investidores no mercado cripto brasileiro.\n"
            "(c) Se o contrato é sobre empresa (MicroStrategy): queda na probabilidade de "
            "venda reforça percepção de que grandes detentores institucionais não estão "
            "liquidando, o que pode ancorar o preço e reduzir volatilidade percebida.\n"
            "Use o ângulo que mais se aplica ao contrato específico."
        ),
    },
    "Comércio": {
        "alta": (
            "ÂNGULO P3 — Comércio/tarifas, probabilidade subindo:\n"
            "Canal direto para o Brasil: tarifas americanas sobre importações chinesas "
            "redirecionam demanda — o Brasil pode se beneficiar como fornecedor alternativo "
            "de soja, carne e commodities, mas o real pode sofrer com aversão ao risco global. "
            "Exportadores brasileiros de commodities e o agronegócio são os mais afetados."
        ),
        "queda": (
            "ÂNGULO P3 — Comércio/tarifas, probabilidade caindo:\n"
            "Menor probabilidade de escalada comercial reduz aversão ao risco global. "
            "Canal: fluxo de capital para emergentes pode aumentar, aliviando pressão "
            "sobre o real. Empresas brasileiras exportadoras voltam a ter visibilidade "
            "sobre demanda chinesa."
        ),
    },
}

# Fallback para categorias não mapeadas
_ANGULO_P3_DEFAULT = {
    "alta": (
        "ÂNGULO P3 — conecte o movimento de alta ao impacto mais direto para o Brasil: "
        "câmbio (real × dólar), custo do crédito doméstico, decisão do Copom, ou Ibovespa. "
        "Escolha o canal mais relevante para este evento específico e conecte ao valor da variação."
    ),
    "queda": (
        "ÂNGULO P3 — conecte o movimento de queda ao impacto mais direto para o Brasil: "
        "câmbio (real × dólar), custo do crédito doméstico, decisão do Copom, ou Ibovespa. "
        "Escolha o canal mais relevante para este evento específico e conecte ao valor da variação."
    ),
}

# ─────────────────────────────────────────────────────────────────
# FEW-SHOT P3 para modelos Llama/Groq — uso opcional
# ─────────────────────────────────────────────────────────────────
# Se quiser forçar o Llama a internalizar o padrão, adicione estes
# exemplos ao início do SYSTEM_PROMPT ou como turn de sistema extra.
# Eles cobrem os 3 casos mais comuns de D3 genérico no Llama:
# cripto preço-alvo alto, cripto queda, FOMC/câmbio.
#
# USO:
#   messages = [
#       {"role": "system", "content": SYSTEM_PROMPT + "\n\n" + FEW_SHOT_P3_LLAMA},
#       {"role": "user",   "content": prompt}
#   ]
# Não use com o Gemini — ele já segue o SYSTEM_PROMPT com mais fidelidade.

FEW_SHOT_P3_LLAMA = """
══════════════════════════════════════════════════════
EXEMPLOS DE PARÁGRAFO 3 CORRETO E INCORRETO
══════════════════════════════════════════════════════

CASO 1 — Cripto: Bitcoin preço-alvo alto (ex: $95k, $130k subindo de probabilidade)
✗ INCORRETO (genérico): "Para o mercado cripto brasileiro, o movimento pode gerar maior interesse e influenciar exchanges e a regulação da CVM."
✓ CORRETO: "Com cerca de 15 milhões de brasileiros com contas em exchanges como Mercado Bitcoin e Binance Brasil, um aumento de 10 pontos percentuais na chance de o Bitcoin atingir US$ 95.000 tende a estimular novos aportes e elevar o volume de negociação doméstico — especialmente entre investidores que usam BTC como reserva de valor frente à inflação."

CASO 2 — Cripto: probabilidade de queda de preço subindo (ex: Bitcoin $55k ficou mais provável)
✗ INCORRETO (genérico): "Investidores brasileiros de cripto devem acompanhar o desenvolvimento da situação e considerar a regulação da CVM."
✓ CORRETO: "A alta de 6 pontos percentuais na chance de o Bitcoin recuar a US$ 55.000 pode levar à liquidação de posições alavancadas por investidores brasileiros — um efeito amplificado pelo fato de exchanges nacionais permitirem alavancagem de até 10×. Quem mantém BTC como hedge cambial pode revisar posições diante do aumento de incerteza."

CASO 3 — FOMC: probabilidade de corte caindo (Fed menos dovish)
✗ INCORRETO (genérico): "Para o Brasil, a mudança nas expectativas sobre os juros americanos pode afetar o câmbio e os mercados financeiros."
✓ CORRETO: "Com o Fed menos propenso a cortar juros, o dólar tende a se fortalecer contra o real — encarecendo importações de bens industriais e pressionando a inflação doméstica. O Copom, que já mantém a Selic em patamar elevado, terá ainda menos espaço para sinalizar cortes em 2026 sem arriscar nova depreciação cambial."
══════════════════════════════════════════════════════
"""


def _montar_prompt(row):
    zscore_abs    = abs(row["zscore"])
    intensidade   = f"cerca de {zscore_abs:.0f} vezes fora do padrão histórico deste contrato"
    direcao_verbo = "subiu" if row["direcao"] == "alta" else "caiu"
    variacao_abs  = abs(row["variacao"])
    categoria     = row.get("categoria", "FOMC")
    direcao       = row.get("direcao", "alta")

    if row["resultado"] == "Yes":
        resultado_txt = "Sim — o mercado acertou a aposta"
    elif row["resultado"] == "No":
        resultado_txt = "Não — o mercado errou a aposta"
    else:
        resultado_txt = "ainda em aberto"

    # Seleciona o ângulo correto para o P3
    angulo_map = _ANGULO_P3.get(categoria, _ANGULO_P3_DEFAULT)
    angulo_p3  = angulo_map.get(direcao, angulo_map.get("alta", ""))

    return f"""
Dados do sinal:

Categoria: {categoria}
Tema do contrato (integre como oração, NÃO cite entre aspas): {row['questao_pt']}
Contrato original em inglês (só referência interna, NÃO use no texto): {row['questao']}
Data do sinal: {row['data'].date()}

─────────────────────────────────────────────
PARÁGRAFO 1 — campos obrigatórios:

  • Probabilidade antes (prob_antes):  {row['prob_antes']:.1f}%
  • Probabilidade no sinal (prob_yes): {row['prob_yes']:.1f}%
  • Variação (campo canônico):         {direcao_verbo} {variacao_abs:.1f} pontos percentuais

  ⚠ ATENÇÃO: use EXATAMENTE {variacao_abs:.1f} pp como variação.
    NÃO recalcule subtraindo prob_antes de prob_yes — o resultado pode diferir
    porque a variação mede o movimento diário, não a diferença entre os pontos
    de referência de 3 dias. Se você recalcular e obter um número diferente,
    ignore o recalculado e use {variacao_abs:.1f}. NÃO mencione nenhum outro valor.

  • Intensidade: {intensidade}
─────────────────────────────────────────────
PARÁGRAFO 2 — sequência completa:
  {row['prob_antes']:.1f}% → {row['prob_yes']:.1f}% → {row['prob_depois']:.1f}% (3 dias depois)
  Volume total do contrato: US$ {row['volume']:,.0f}
  Resultado final: {resultado_txt}
─────────────────────────────────────────────
{angulo_p3}
─────────────────────────────────────────────

Escreva a notícia seguindo a estrutura do sistema.
Antes de fechar P1: verifique se o tema está integrado como oração (sem aspas).
Antes de fechar P3: substitua qualquer "porque"/"em resposta a" por "pode indicar"/"sugere que".
"""


def chamar_gemini(prompt):
    r = client.models.generate_content(
        model=MODELOS["gemini"]["modelo"],
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=0.4,
            safety_settings=[
                types.SafetySetting(category=c, threshold=types.HarmBlockThreshold.BLOCK_NONE)
                for c in [
                    types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                    types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                    types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                    types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                ]
            ],
        ),
    )
    if r.candidates and r.candidates[0].finish_reason != types.FinishReason.STOP:
        raise Exception(f"Geração interrompida: {r.candidates[0].finish_reason}")
    if not r.text:
        raise Exception("Resposta vazia do Gemini")
    return r.text.strip()


def chamar_groq(prompt):
    resp = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {GROQ_KEY}", "Content-Type": "application/json"},
        json={
            "model": MODELOS["groq_llama"]["modelo"],
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
            "max_tokens": 900, "temperature": 0.4,
        },
        timeout=40,
    )
    data = resp.json()
    if resp.status_code == 429:
        raise Exception(f"Groq rate limit: {data.get('error',{}).get('message','')}")
    if "choices" not in data or not data["choices"]:
        raise Exception(f"Groq sem choices: {data.get('error', data)}")
    conteudo = data["choices"][0]["message"].get("content", "").strip()
    if not conteudo:
        raise Exception("Resposta vazia do Groq")
    return conteudo


def chamar_openrouter(prompt, modelo_id):
    cfg  = MODELOS[modelo_id]
    resp = requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_KEY}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://signal-tcc.dev",
            "X-Title": "Signal TCC",
        },
        json={
            "model": cfg["modelo"],
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
            "max_tokens": 900, "temperature": 0.4,
        },
        timeout=40,
    )
    data = resp.json()
    if "choices" not in data or not data["choices"]:
        raise Exception(f"OpenRouter sem choices: {data.get('error', data)}")
    conteudo = data["choices"][0]["message"].get("content", "").strip()
    if not conteudo:
        raise Exception("Resposta vazia do OpenRouter")
    return conteudo


def normalizar_narrativa(texto: str) -> str:
    if not texto or texto.startswith("ERRO"):
        return texto
    texto = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', texto)
    texto = re.sub(r'\*(.*?)\*',     r'<b>\1</b>', texto)
    if "<p>" in texto or "</p>" in texto:
        texto = re.sub(r'</p>\s*</p>', '</p>', texto)
        texto = re.sub(r'<p>\s*<p>',   '<p>',  texto)
    else:
        texto = re.sub(r'\n{2,}', '</p><p>', texto)
    if not texto.startswith("<p>"):
        texto = "<p>" + texto
    if not texto.endswith("</p>"):
        texto = texto + "</p>"
    texto = re.sub(r'<p>\s*</p>', '', texto)
    return texto.strip()


def waterfall(prompt):
    ordem = [
        ("gemini",     lambda p: chamar_gemini(p)),
        ("groq_llama", lambda p: chamar_groq(p)),
        ("gemma",      lambda p: chamar_openrouter(p, "gemma")),
        ("qwen",       lambda p: chamar_openrouter(p, "qwen")),
    ]
    tentativas = []
    for modelo_id, fn in ordem:
        nome = MODELOS[modelo_id]["nome"]
        try:
            narrativa = fn(prompt)
            tentativas.append({"modelo": nome, "status": "ok"})
            return narrativa, nome, tentativas
        except Exception as e:
            tentativas.append({"modelo": nome, "status": f"erro: {e}"})
            print(f"  [{nome}] falhou → tentando próximo...")
            time.sleep(1)
    return "ERRO: todos os modelos falharam.", "nenhum", tentativas


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 5 — Sistema de pontuação das narrativas
# ═══════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────
# pontuar_narrativa()  — ajuste para penalizar recálculo (FIX-I)
# Substitui a versão do v2.
# Única mudança: adiciona detecção do padrão "não, caiu/subiu X"
# que indica autocorreção visível do modelo.
# ─────────────────────────────────────────────────────────────────

import re

def pontuar_narrativa(narrativa: str, row) -> tuple:
    if not narrativa or narrativa.startswith("ERRO"):
        return 0, {"erro": narrativa}

    texto          = narrativa.lower()
    texto_original = narrativa
    scores         = {}

    def num_presente(valor, txt):
        candidatos = set()
        try:
            v = float(valor)
            for fmt in [str(int(v)), str(round(v,1)), f"{v:.1f}", f"{v:.0f}", str(v).replace(".0","")]:
                candidatos.add(fmt)
        except (ValueError, TypeError):
            candidatos.add(str(valor))
        return any(c in txt for c in candidatos)

    p = 0
    if num_presente(row["prob_yes"], texto_original):      p += 10
    if num_presente(abs(row["variacao"]), texto_original): p += 10
    if str(row["direcao"]).lower() in texto:               p += 5
    if str(row["magnitude"]).lower() in texto:             p += 5
    scores["ancoragem_factual"] = p

    n_p = len(re.findall(r'<p>', texto_original))
    if n_p == 0:
        n_p = max(1, len(texto_original.split("\n\n")))
    scores["extensao"] = 25 if 2 <= n_p <= 3 else (15 if n_p in [1, 4] else 5)

    termos = [
        "federal reserve", "fed", "juros", "taxa", "política monetária",
        "mercado", "probabilidade", "investidor", "economia", "corte",
        "inflação", "recessão", "brasil", "câmbio", "bps", "fomc",
        "emergente", "dólar", "monetary", "interest", "copom", "ibovespa",
        "real", "crédito", "banco central", "bitcoin", "cripto", "cpi",
        "pib", "desemprego", "tarifa",
    ]
    scores["contexto_economico"] = min(25, sum(1 for t in termos if t in texto) * 3)

    jargao = ["z-score", "brier score", "standard deviation",
              "changepoint", "zscore", "fidelity", "sliding window"]
    abertura_proib = texto_original.lower().lstrip("<p>").startswith(
        ("o signal detectou", "um movimento foi detectado", "uma anomalia foi")
    )
    # FIX-F: penaliza título entre aspas (padrão do Gemini)
    titulo_entre_aspas = bool(re.search(
        r'["""«»\u201c\u201d\u2018\u2019][^"""«»\u201c\u201d\u2018\u2019]{5,80}[?!]["""«»\u201c\u201d\u2018\u2019]',
        texto_original
    ))
    # FIX-I: penaliza autocorreção visível ("não, caiu X" / "não, subiu X")
    autocorrecao_visivel = bool(re.search(
        r'\bnão[,.]?\s+(caiu|subiu|aumentou|diminuiu|caindo|subindo)\b',
        texto_original, re.IGNORECASE
    ))
    penalidade = (sum(4 for j in jargao if j in texto)
                  + (10 if abertura_proib else 0)
                  + (5  if titulo_entre_aspas else 0)
                  + (8  if autocorrecao_visivel else 0))   # nova penalidade FIX-I
    scores["clareza"] = max(0, 20 - penalidade)

    return sum(scores.values()), scores


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 6 — Pipeline principal
# ═══════════════════════════════════════════════════════════════════

def calcular_brier_score(eventos: list):
    validos = [e for e in eventos if e.get("resultado") in ["Yes", "No"]]
    if not validos:
        return None
    soma = sum(((e["prob_yes"]/100) - (1 if e["resultado"]=="Yes" else 0))**2 for e in validos)
    return round(soma / len(validos), 4)


def rodar_pipeline(mercados_coletados, limiar_z=None, max_novos=20, df_sensibilidade=None):
    limiar_z = limiar_z or PARAMS["limiar_z_padrao"]

    print(f"\nDetectando signals com z* = {limiar_z}...")
    todos = [s for m in mercados_coletados
             for s in [detectar_signals(m, limiar_z=limiar_z)] if not s.empty]
    if not todos:
        print("Nenhum signal detectado.")
        return []

    df_dedup = deduplicar_signals(pd.concat(todos, ignore_index=True))
    print(f"Signals únicos: {len(df_dedup)}")

    # Mostra distribuição de magnitude para validação
    mags = df_dedup["magnitude"].value_counts().to_dict()
    print(f"  Distribuição de magnitude: {mags}")
    cats = df_dedup.get("categoria", pd.Series()).value_counts().to_dict() if "categoria" in df_dedup else {}
    if cats:
        print(f"  Por categoria: {cats}")

    processados = carregar_processados()
    df_novos = df_dedup[
        ~df_dedup.apply(lambda r: signal_id(r) in processados, axis=1)
    ].head(max_novos)
    print(f"Novos para processar: {len(df_novos)} (já processados: {len(df_dedup)-len(df_novos)})")

    if df_novos.empty:
        print("Feed já atualizado.")
        return _carregar_feed()

    feed = _carregar_feed()

    for idx, (_, row) in enumerate(df_novos.iterrows(), 1):
        sid        = signal_id(row)
        questao_pt = traduzir_questao(row["questao"])
        print(f"\n[{idx}/{len(df_novos)}] {row['data'].date()} z={row['zscore']:.2f} [{row.get('categoria','?')}] {row['questao'][:50]}")
        print(f"  PT: {questao_pt}")

        row_com_pt           = row.copy()
        row_com_pt["questao_pt"] = questao_pt

        prompt    = _montar_prompt(row_com_pt)
        narrativa, modelo_usado, tentativas = waterfall(prompt)
        narrativa = normalizar_narrativa(narrativa)
        score, detalhes = pontuar_narrativa(narrativa, row)

        z_abs = abs(float(row["zscore"]))
        entrada = {
            "id"                  : sid,
            "data"                : str(row["data"].date()),
            "questao"             : row["questao"],
            "questao_pt"          : questao_pt,
            "evento"              : row["evento"],
            "categoria"           : row.get("categoria", "FOMC"),
            "prob_yes"            : round(float(row["prob_yes"]), 1),
            "variacao"            : round(float(row["variacao"]), 2),
            "zscore"              : round(float(row["zscore"]), 2),
            "magnitude"           : row["magnitude"],
            "magnitude_descricao" : magnitude_descricao(row["magnitude"], z_abs),  # FIX-A
            "direcao"             : row["direcao"],
            "prob_antes"          : round(float(row["prob_antes"]), 1),
            "prob_depois"         : round(float(row["prob_depois"]), 1),
            "volume"              : float(row["volume"]),
            "end_date"            : row["end_date"],
            "resultado"           : row["resultado"],
            "narrativa"           : narrativa,
            "modelo_usado"        : modelo_usado,
            "score"               : score,
            "score_detalhes"      : detalhes,
            "tentativas"          : tentativas,
            "token_id"            : row["token_id"],
            "gerado_em"           : datetime.now(timezone.utc).isoformat(),
        }

        feed.append(entrada)
        processados.add(sid)
        salvar_processados(processados)
        print(f"  Modelo: {modelo_usado} | Score: {score}/100")
        time.sleep(3)

    feed.sort(key=lambda x: x["data"], reverse=True)
    brier = calcular_brier_score(feed)
    _salvar_feed(feed, brier_score=brier, df_sensibilidade=df_sensibilidade)

    print(f"\n✓ Feed atualizado: {len(feed)} eventos totais")
    if brier is not None:
        n_val = sum(1 for e in feed if e.get("resultado") in ["Yes","No"])
        print(f"  Brier Score: {brier:.4f} (sobre {n_val} eventos com desfecho)")
    return feed


def _carregar_feed():
    if os.path.exists(FEED_FILE):
        with open(FEED_FILE, "r", encoding="utf-8") as f:
            raw = json.load(f)
        eventos = raw.get("events", [])
        vistos = {}
        for e in eventos:
            eid = e.get("id","")
            if eid not in vistos or e.get("gerado_em","") > vistos[eid].get("gerado_em",""):
                vistos[eid] = e
        return list(vistos.values())
    return []


def _salvar_feed(eventos, brier_score=None, df_sensibilidade=None):
    vistos = {}
    for e in eventos:
        eid = e.get("id","")
        if eid not in vistos or e.get("gerado_em","") > vistos[eid].get("gerado_em",""):
            vistos[eid] = e
    eventos_unicos = sorted(vistos.values(), key=lambda x: x["data"], reverse=True)
    sens_list = df_sensibilidade.to_dict(orient="records") if df_sensibilidade is not None and not df_sensibilidade.empty else None
    payload = {
        "gerado_em"    : datetime.now(timezone.utc).isoformat(),
        "threshold"    : PARAMS["limiar_z_padrao"],
        "janela"       : PARAMS["janela_zscore"],
        "brier_score"  : brier_score,
        "sensibilidade": sens_list,
        "events"       : eventos_unicos,
    }
    with open(FEED_FILE, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)


def exportar_serie_para_frontend(mercados_coletados, token_id_alvo: str):
    for m in mercados_coletados:
        if m["token_id"] != token_id_alvo:
            continue
        df = m["df"].copy()
        df["variacao"] = df["prob_yes"].diff()
        vol = (df["variacao"].rolling(PARAMS["janela_zscore"], min_periods=5)
                             .std().bfill().clip(lower=PARAMS["vol_piso"]))
        df["zscore"] = (df["variacao"] / vol).round(2)
        df["dia"]    = df["data"].dt.strftime("%d/%m")
        df_diario    = df.groupby("dia", sort=False).last().reset_index()
        if not os.path.exists(FEED_FILE):
            print(f"ERRO: {FEED_FILE} não encontrado.")
            return
        with open(FEED_FILE, "r", encoding="utf-8") as f:
            payload = json.load(f)
        payload["serie"] = {
            "token_id"  : token_id_alvo,
            "questao"   : m["questao"],
            "questao_pt": traduzir_questao(m["questao"]),
            "dates"     : df_diario["dia"].tolist(),
            "probs"     : df_diario["prob_yes"].round(1).tolist(),
            "zscores"   : df_diario["zscore"].fillna(0).tolist(),
        }
        with open(FEED_FILE, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2, default=str)
        print(f"✓ Série exportada: {len(df_diario)} pontos → {FEED_FILE}")
        print(f"  Contrato PT: {traduzir_questao(m['questao'])}")
        return
    print(f"ERRO: token_id '{token_id_alvo}' não encontrado.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CÉLULA 7 — Execução
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # Descomente se precisar reprocessar IDs específicos:
    # forcar_reprocessamento({"id1", "id2"})

    print("=== PASSO 1: Coleta de dados ===")
    mercados_coletados = coletar_mercados_fomc()

    print("\n=== PASSO 2: Análise de sensibilidade ===")
    df_sensibilidade = analise_sensibilidade(mercados_coletados)
    df_sensibilidade.to_csv("sensibilidade.csv", index=False)
    print("  → Salvo em sensibilidade.csv")

    print("\n=== PASSO 3: Pipeline de narrativas ===")
    feed = rodar_pipeline(
        mercados_coletados,
        limiar_z=2.0,
        max_novos=20,
        df_sensibilidade=df_sensibilidade,
    )

    if feed:
        print("\n=== PASSO 4: Exportar série temporal ===")
        tokens_mercados = {m["token_id"]: m for m in mercados_coletados}
        alvo = max(
            (ev for ev in feed if ev.get("token_id") in tokens_mercados),
            key=lambda ev: tokens_mercados[ev["token_id"]]["volume"],
            default=None,
        )
        if alvo:
            exportar_serie_para_frontend(mercados_coletados, alvo["token_id"])

    print(f"\n✓ Concluído. Arquivo: {FEED_FILE}")
    with open(FEED_FILE, "r") as f:
        check = json.load(f)
    cats = {}
    mags = {}
    for e in check.get("events", []):
        c = e.get("categoria","?"); cats[c] = cats.get(c,0)+1
        mg = e.get("magnitude","?"); mags[mg] = mags.get(mg,0)+1
    print(f"\nVerificação do {FEED_FILE}:")
    print(f"  Eventos      : {len(check.get('events',[]))}")
    print(f"  Por categoria: {cats}")
    print(f"  Magnitudes   : {mags}")
    print(f"  Brier Score  : {check.get('brier_score','—')}")
    print(f"  Sensibilidade: {'sim (' + str(len(check['sensibilidade'])) + ' thresholds)' if check.get('sensibilidade') else 'não'}")
    print(f"  Serie        : {'sim (' + str(len(check['serie']['dates'])) + ' pontos)' if check.get('serie') else 'não'}")
    print(f"  Threshold    : {check.get('threshold')}")
    print(f"  Gerado em    : {check.get('gerado_em')}")

In [ ]:
# # ╔══════════════════════════════════════════════════════════════════╗
# # ║  SIGNAL — Backend v1                                            ║
# # ║  Sistema de inteligência macroeconômica via mercados preditivos ║
# # ║  TCC · Centro de Informática · 2026                             ║
# # ╚══════════════════════════════════════════════════════════════════╝
# #
# # CORREÇÕES v1 frente ao test_version:
# #
# #   FIX-A  Escala de magnitude mantida e documentada no manuscrito.
# #          A análise dos z-scores reais (3.73–3.83 para o dataset FOMC)
# #          confirma que TODOS os sinais caem na faixa "Significativo"
# #          com z*=2.0. Isso NÃO é um bug — é uma propriedade do dataset:
# #          contratos FOMC têm volatilidade histórica homogênea, o que
# #          comprova que o z-score detecta variações genuinamente anômalas
# #          (3–4× acima do normal). As categorias Forte (4.0–5.0) e Raro
# #          (≥5.0) aparecerão naturalmente quando o escopo incluir cripto,
# #          CPI ou eventos de maior volatilidade. A escala relativa ao z*
# #          (FIX-1 do test_version) é mantida. O campo "magnitude_descricao" é
# #          adicionado ao payload para o dashboard mostrar rótulos
# #          contextualizados por z*.
# #
# #   FIX-B  Título do contrato não mais citado entre aspas no corpo.
# #          O SYSTEM_PROMPT agora instrui explicitamente a integrar a
# #          pergunta do contrato como oração subordinada ("a probabilidade
# #          de que o Fed corte..."), nunca como citação entre aspas.
# #          Exemplos concretos de integração correta adicionados ao prompt.
# #
# #   FIX-C  Variação em pp reforçada no prompt para elevar ancoragem.
# #          3 eventos tiveram ancoragem_factual = 10–15/30 porque a LLM
# #          não mencionou o valor exato da variação. O _montar_prompt()
# #          agora destaca a variação como campo obrigatório separado e
# #          adiciona instrução explícita no prompt para citá-la no P1.
# #
# #   FIX-D  Coleta expandida para além do escopo FOMC.
# #          Os termos "cpi", "inflation", "bitcoin" etc. não batiam com
# #          os títulos reais da Polymarket. Causa: a API /events filtra
# #          por substring no título. Títulos reais são: "Will CPI be
# #          above X%?", "Bitcoin price end of year?", "US recession 2026?".
# #          Fix: termos ampliados com variantes curtas ("cpi ", "bitcoin",
# #          "recession", "unemployment") + busca separada por categoria
# #          com parâmetros independentes. Aumentado min_vol para categorias
# #          novas (US$ 200k) para garantir liquidez mínima.
# #
# #   FIX-E  max_mercados_por_execucao removido como limite principal.
# #          O limite anterior (30) cortava mercados antes de coletar
# #          categorias não-FOMC. Substituído por max_eventos_por_categoria
# #          (5 por categoria) que garante diversidade sem sobrecarregar a API.


# import requests, json, time, os, hashlib, re
# import pandas as pd
# import numpy as np
# from datetime import datetime, timezone
# from google import genai
# from google.genai import types

# client = genai.Client(api_key=GEMINI_KEY)

# BASE     = "https://gamma-api.polymarket.com"
# CLOB     = "https://clob.polymarket.com"
# START_TS = 1704067200

# PARAMS = {
#     "janela_zscore"            : 14,
#     "limiar_z_padrao"          : 2.0,
#     "var_minima_pp"            : 3.0,
#     "vol_piso"                 : 1.0,
#     "gap_dedup_dias"           : 7,
#     "thresholds_sens"          : [1.5, 2.0, 2.5],
#     "max_eventos_por_categoria": 5,    # FIX-E: diversidade sem sobrecarga
# }

# CACHE_FILE = "processados.json"
# FEED_FILE  = "demo_cache.json"
# COLETA_LOG = "coleta_log.json"


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 0b — Utilitário: forçar reprocessamento de IDs
# # ═══════════════════════════════════════════════════════════════════

# IDS_REPROCESSAR = set()  # adicione IDs aqui para forçar reprocessamento

# def forcar_reprocessamento(ids: set = None):
#     ids = ids or IDS_REPROCESSAR
#     if not ids:
#         print("Nenhum ID para reprocessar.")
#         return
#     if not os.path.exists(CACHE_FILE):
#         print("processados.json não encontrado.")
#         return
#     with open(CACHE_FILE, "r") as f:
#         processados = set(json.load(f))
#     antes = len(processados)
#     processados -= ids
#     with open(CACHE_FILE, "w") as f:
#         json.dump(list(processados), f)
#     print(f"Removidos {antes - len(processados)} IDs do cache.")


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 1 — Funções base e coleta de dados
# # ═══════════════════════════════════════════════════════════════════

# def _get_com_retry(url, params=None, max_tentativas=3, timeout=15):
#     for tentativa in range(max_tentativas):
#         try:
#             r = requests.get(url, params=params, timeout=timeout)
#             if r.status_code == 200:
#                 return r
#             if r.status_code in (429, 503):
#                 espera = 2 ** tentativa * 5
#                 print(f"  Rate limit ({r.status_code}) — aguardando {espera}s...")
#                 time.sleep(espera)
#                 continue
#             return r
#         except requests.exceptions.RequestException as e:
#             if tentativa < max_tentativas - 1:
#                 time.sleep(3)
#             else:
#                 print(f"  ERRO de rede: {e}")
#                 return None
#     return None


# def extrair_token_ids(m):
#     raw = m.get("clobTokenIds", [])
#     if isinstance(raw, list):
#         return raw
#     try:
#         return json.loads(raw)
#     except Exception:
#         return []


# def extrair_resultado(m):
#     op = m.get("outcomePrices")
#     if not op or op in [None, "null", ""]:
#         return None, None
#     try:
#         p = json.loads(op)
#         if not p:
#             return None, None
#         p0 = float(p[0])
#         if p0 == 1.0:   return 1, "Yes"
#         elif p0 == 0.0: return 0, "No"
#         return None, None
#     except Exception:
#         return None, None


# def categorizar_evento(titulo: str, questao: str) -> str:
#     texto = (titulo + " " + questao).lower()
#     if any(t in texto for t in ["fomc", "fed rate", "fed decision", "fed interest", "federal funds", "interest rate"]):
#         return "FOMC"
#     if any(t in texto for t in ["inflation", "cpi", "pce", "consumer price"]):
#         return "Inflação"
#     if any(t in texto for t in ["gdp", "recession", "unemployment", "jobs", "payroll"]):
#         return "Crescimento"
#     if any(t in texto for t in ["bitcoin", "crypto", "ethereum", "btc"]):
#         return "Cripto"
#     if any(t in texto for t in ["tariff", "trade war", "trade deal"]):
#         return "Comércio"
#     return "Outro"


# def buscar_historico(token_id):
#     r = _get_com_retry(f"{CLOB}/prices-history", params={
#         "market"  : token_id,
#         "interval": "1d",
#         "fidelity": 100,
#         "startTs" : START_TS,
#     })
#     if r is None or r.status_code != 200:
#         return pd.DataFrame()
#     hist = r.json().get("history", [])
#     if not hist:
#         return pd.DataFrame()
#     df = pd.DataFrame(hist)
#     df["data"]     = pd.to_datetime(df["t"], unit="s")
#     df["prob_yes"] = df["p"].astype(float) * 100
#     return df[["data", "prob_yes"]].sort_values("data").reset_index(drop=True)


# # FIX-D: termos expandidos com variantes reais dos títulos da Polymarket
# # Cada categoria tem termos curtos que batem como substring nos títulos reais
# TERMOS_POR_CATEGORIA = {
#     "FOMC": [
#         "fed decision", "fed interest", "fed rate", "fomc",
#         "federal funds", "interest rate cut", "rate hike",
#     ],
#     "Inflação": [
#         "cpi", "inflation", "pce", "consumer price index",
#     ],
#     "Crescimento": [
#         "recession", "gdp", "unemployment", "nonfarm", "payroll",
#     ],
#     "Cripto": [
#         "bitcoin", "btc ", "ethereum", "crypto",
#     ],
#     "Comércio": [
#         "tariff", "trade war", "us china",
#     ],
# }

# def coletar_mercados_fomc():
#     """
#     FIX-D + FIX-E: coleta por categoria com limite por categoria,
#     garantindo diversidade sem sobrecarregar a API.
#     """
#     max_por_cat = PARAMS["max_eventos_por_categoria"]
#     coletados   = []
#     vistos_ids  = set()
#     log_coleta  = {"timestamp": datetime.now(timezone.utc).isoformat(), "eventos": [], "por_categoria": {}}

#     for categoria, termos in TERMOS_POR_CATEGORIA.items():
#         eventos_cat = []

#         for params_busca in [
#             {"limit": 100, "active": "false", "closed": "true",  "order": "volume", "ascending": "false"},
#             {"limit": 100, "active": "true",  "closed": "false", "order": "volume", "ascending": "false"},
#         ]:
#             r = _get_com_retry(f"{BASE}/events", params=params_busca)
#             if r is None or r.status_code != 200:
#                 continue
#             min_vol = 500_000 if params_busca["closed"] == "true" else 200_000
#             for e in r.json():
#                 eid    = e.get("id", "")
#                 titulo = e.get("title", "").lower()
#                 vol    = float(e.get("volume", 0))
#                 if eid in vistos_ids:
#                     continue
#                 if any(t in titulo for t in termos) and vol >= min_vol:
#                     eventos_cat.append(e)
#                     vistos_ids.add(eid)

#         # Limita por categoria para garantir diversidade
#         eventos_cat = eventos_cat[:max_por_cat]
#         log_coleta["por_categoria"][categoria] = len(eventos_cat)

#         for evento in eventos_cat:
#             eid  = evento.get("id", "")
#             r_ev = _get_com_retry(f"{BASE}/events/{eid}")
#             if r_ev is None or r_ev.status_code != 200:
#                 continue

#             for m in r_ev.json().get("markets", []):
#                 tids = extrair_token_ids(m)
#                 if not tids:
#                     continue
#                 res_bin, res_str = extrair_resultado(m)
#                 df_hist          = buscar_historico(tids[0])
#                 if df_hist.empty or len(df_hist) < 5:
#                     continue
#                 coletados.append({
#                     "evento"       : evento.get("title", ""),
#                     "questao"      : m.get("question", ""),
#                     "token_id"     : tids[0],
#                     "end_date"     : m.get("endDateIso", "")[:10],
#                     "resultado"    : res_str,
#                     "resultado_bin": res_bin,
#                     "volume"       : float(m.get("volumeNum", 0)),
#                     "categoria"    : categoria,
#                     "df"           : df_hist,
#                 })

#             log_coleta["eventos"].append({"id": eid, "titulo": evento.get("title",""), "categoria": categoria})
#             time.sleep(0.2)

#     with open(COLETA_LOG, "w", encoding="utf-8") as f:
#         json.dump(log_coleta, f, ensure_ascii=False, indent=2)

#     print(f"Eventos encontrados: {sum(log_coleta['por_categoria'].values())}")
#     print(f"  Por categoria: {log_coleta['por_categoria']}")
#     print(f"Mercados com histórico: {len(coletados)}")
#     return coletados


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 1b — Tradução automática dos títulos dos contratos
# # ═══════════════════════════════════════════════════════════════════

# _MESES_EN_PT = {
#     "January": "janeiro", "February": "fevereiro", "March": "março",
#     "April": "abril",     "May": "maio",            "June": "junho",
#     "July": "julho",      "August": "agosto",       "September": "setembro",
#     "October": "outubro", "November": "novembro",   "December": "dezembro",
# }

# def _mes_pt(texto: str) -> str:
#     for en, pt in _MESES_EN_PT.items():
#         texto = texto.replace(en, pt)
#     return texto


# def traduzir_questao(questao: str) -> str:
#     q = questao.strip()

#     # Padrão A: "No change in Fed interest rates after YYYY Month meeting?"
#     m = re.match(r"No change in Fed interest rates after (\d{4}) (\w+) meeting\?", q, re.I)
#     if m:
#         return f"O Fed vai manter os juros sem alteração na reunião de {_mes_pt(m.group(2))} de {m.group(1)}?"

#     # Padrão B: "No change in Fed interest rates after Month YYYY meeting?"
#     m = re.match(r"No change in Fed interest rates after (\w+ \d{4}) meeting\?", q, re.I)
#     if m:
#         return f"O Fed vai manter os juros sem alteração na reunião de {_mes_pt(m.group(1))}?"

#     # Padrão C: "Fed decreases interest rates by X[+] bps after Month YYYY meeting?"
#     m = re.match(r"Fed decreases interest rates by (\d+)(\+?) bps after (\w+ \d{4}) meeting\?", q, re.I)
#     if m:
#         sufixo = " ou mais" if m.group(2) else ""
#         return f"O Fed vai cortar os juros em {m.group(1)} pontos-base{sufixo} na reunião de {_mes_pt(m.group(3))}?"

#     # Padrão D: "Will the Fed decrease interest rates by X[+] bps after the Month YYYY meeting?"
#     m = re.match(r"Will the Fed decrease interest rates by (\d+)(\+?) bps after the (\w+ \d{4}) meeting\?", q, re.I)
#     if m:
#         sufixo = " ou mais" if m.group(2) else ""
#         return f"O Fed vai cortar os juros em {m.group(1)} pontos-base{sufixo} na reunião de {_mes_pt(m.group(3))}?"

#     # Padrão E: "Will N Fed rate cuts happen in YYYY?"
#     m = re.match(r"Will (\d+) Fed rate cuts happen in (\d{4})\?", q, re.I)
#     if m:
#         return f"O Fed vai realizar {m.group(1)} cortes de juros em {m.group(2)}?"

#     # Padrão F: "Will the upper bound ... be X% at the end of YYYY?"
#     m = re.match(r"Will the upper bound of the target federal funds rate be ([\d.]+)% at the end of (\d{4})\?", q, re.I)
#     if m:
#         return f"A taxa de juros americana vai terminar {m.group(2)} em {m.group(1)}% ao ano?"

#     # Padrão G: "Will the upper bound ... be ≥ X% at the end of YYYY?"
#     m = re.match(r"Will the upper bound of the target federal funds rate be ≥\s*([\d.]+)%\s*at the end of (\d{4})\?", q, re.I)
#     if m:
#         return f"A taxa de juros americana vai terminar {m.group(2)} em {m.group(1)}% ao ano ou mais?"

#     # Padrão H: CPI / inflação
#     m = re.match(r"Will (US |the )?(CPI|inflation)(.*?)(\d{4})\?", q, re.I)
#     if m:
#         return f"A inflação americana vai {m.group(3).strip()} em {m.group(4)}?"

#     # Padrão I: Recessão
#     m = re.match(r"Will (the US|the United States|US) (enter a )?recession.*?(\d{4})\?", q, re.I)
#     if m:
#         return f"Os EUA vão entrar em recessão em {m.group(3)}?"

#     # Padrão J: Bitcoin
#     m = re.match(r"Will Bitcoin(.*?)(reach|hit|exceed|be above)\s*\$?([\d,]+)(.*?)\?", q, re.I)
#     if m:
#         return f"O Bitcoin vai atingir US$ {m.group(3).replace(',','')}?"

#     return questao  # fallback: retorna original


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 2 — Detector de anomalias contextual
# # ═══════════════════════════════════════════════════════════════════

# def classificar_magnitude(z_abs: float, limiar_z: float) -> str:
#     """
#     Classificação relativa ao limiar z* (mantida do test_version — FIX-A).
#     Escala com z*=2.0:
#       Moderado     : 2.0 ≤ |z| < 3.0
#       Significativo: 3.0 ≤ |z| < 4.0  ← faixa do dataset FOMC atual
#       Forte        : 4.0 ≤ |z| < 5.0  ← esperado ao expandir para cripto/CPI
#       Raro         : |z| ≥ 5.0
#     """
#     if z_abs < limiar_z * 1.5:   return "Moderado"
#     elif z_abs < limiar_z * 2.0: return "Significativo"
#     elif z_abs < limiar_z * 2.5: return "Forte"
#     else:                         return "Raro"


# def magnitude_descricao(magnitude: str, z_abs: float) -> str:
#     """
#     FIX-A: Descrição textual para o dashboard, contextualizada pelo valor real do z.
#     """
#     desc = {
#         "Moderado"     : f"Movimento incomum (z={z_abs:.1f}×) — fora do padrão, mas ainda dentro da faixa habitual de eventos macro",
#         "Significativo": f"Sinal relevante (z={z_abs:.1f}×) — variação {z_abs:.1f} vezes acima do comportamento normal deste contrato",
#         "Forte"        : f"Sinal forte (z={z_abs:.1f}×) — movimento raro, típico de notícias de alto impacto",
#         "Raro"         : f"Sinal extremo (z={z_abs:.1f}×) — evento estatisticamente excepcional neste mercado",
#     }
#     return desc.get(magnitude, "")


# def detectar_signals(mercado, janela=None, limiar_z=None, var_minima=None):
#     janela     = janela     or PARAMS["janela_zscore"]
#     limiar_z   = limiar_z   or PARAMS["limiar_z_padrao"]
#     var_minima = var_minima or PARAMS["var_minima_pp"]

#     df = mercado["df"].copy().sort_values("data").reset_index(drop=True)
#     if len(df) < janela + 2:
#         return pd.DataFrame()

#     df["variacao"]         = df["prob_yes"].diff()
#     df["vol_local"]        = (df["variacao"]
#                                .rolling(janela, min_periods=max(5, janela // 2))
#                                .std().bfill())
#     df["vol_local_segura"] = df["vol_local"].clip(lower=PARAMS["vol_piso"])
#     df["zscore"]           = df["variacao"] / df["vol_local_segura"]

#     eventos = df[
#         (df["zscore"].abs() >= limiar_z) &
#         (df["variacao"].abs() >= var_minima)
#     ].copy()
#     if eventos.empty:
#         return pd.DataFrame()

#     eventos["magnitude"] = eventos["zscore"].apply(lambda z: classificar_magnitude(abs(z), limiar_z))
#     eventos["direcao"]   = eventos["zscore"].apply(lambda z: "alta" if z > 0 else "queda")
#     eventos["evento"]    = mercado["evento"]
#     eventos["questao"]   = mercado["questao"]
#     eventos["volume"]    = mercado["volume"]
#     eventos["end_date"]  = mercado["end_date"]
#     eventos["resultado"] = mercado["resultado"]
#     eventos["token_id"]  = mercado["token_id"]
#     eventos["categoria"] = mercado.get("categoria", "FOMC")

#     for i in eventos.index:
#         idx = df.index.get_loc(i)
#         eventos.loc[i, "prob_antes"]  = df["prob_yes"].iloc[max(0, idx - 3)]
#         eventos.loc[i, "prob_depois"] = df["prob_yes"].iloc[min(len(df) - 1, idx + 3)]

#     return eventos[[
#         "data", "prob_yes", "variacao", "zscore", "magnitude", "direcao",
#         "prob_antes", "prob_depois", "evento", "questao", "volume",
#         "end_date", "resultado", "token_id", "categoria",
#     ]]


# def signal_id(row):
#     key = f"{row['token_id']}_{str(row['data'].date())}"
#     return hashlib.md5(key.encode()).hexdigest()[:12]


# def deduplicar_signals(df_signals, gap_dias=None):
#     gap_dias = gap_dias or PARAMS["gap_dedup_dias"]
#     df = df_signals.copy()
#     df["data"] = pd.to_datetime(df["data"])
#     selecionados = []
#     for _, grupo in df.groupby("evento"):
#         grupo  = grupo.sort_values("zscore", key=abs, ascending=False)
#         usados = []
#         for _, row in grupo.iterrows():
#             if not any(abs((row["data"] - d).days) <= gap_dias for d in usados):
#                 selecionados.append(row)
#                 usados.append(row["data"])
#     return (pd.DataFrame(selecionados)
#               .sort_values("zscore", key=abs, ascending=False)
#               .reset_index(drop=True))


# def carregar_processados():
#     if os.path.exists(CACHE_FILE):
#         with open(CACHE_FILE, "r") as f:
#             return set(json.load(f))
#     return set()


# def salvar_processados(ids: set):
#     with open(CACHE_FILE, "w") as f:
#         json.dump(list(ids), f)


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 3 — Análise de sensibilidade do threshold
# # ═══════════════════════════════════════════════════════════════════

# def analise_sensibilidade(mercados_coletados, thresholds=None):
#     thresholds = thresholds or PARAMS["thresholds_sens"]
#     resumo = []
#     for z in thresholds:
#         todos = [s for m in mercados_coletados
#                  for s in [detectar_signals(m, limiar_z=z)] if not s.empty]
#         if not todos:
#             resumo.append({"z_threshold": z, "signals_brutos": 0, "signals_dedup": 0,
#                            "moderados": 0, "significativos": 0, "fortes_raros": 0, "alto_volume": 0})
#             continue
#         df_raw  = pd.concat(todos, ignore_index=True)
#         df_deup = deduplicar_signals(df_raw)
#         resumo.append({
#             "z_threshold"   : z,
#             "signals_brutos": len(df_raw),
#             "signals_dedup" : len(df_deup),
#             "moderados"     : (df_deup["magnitude"] == "Moderado").sum(),
#             "significativos": (df_deup["magnitude"] == "Significativo").sum(),
#             "fortes_raros"  : df_deup["magnitude"].isin(["Forte", "Raro"]).sum(),
#             "alto_volume"   : (df_deup["volume"] >= 1_000_000).sum(),
#         })
#     df_sens = pd.DataFrame(resumo)
#     print("\n── Análise de Sensibilidade do Threshold ──────────────────")
#     print(df_sens.to_string(index=False))
#     for _, row in df_sens.iterrows():
#         ratio = row["alto_volume"] / row["signals_dedup"] if row["signals_dedup"] > 0 else 0
#         print(f"  z* = {row['z_threshold']:.1f} → razão precisão-proxy = {ratio:.2f}")
#     return df_sens


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 4 — Pipeline de geração de narrativas
# # ═══════════════════════════════════════════════════════════════════

# MODELOS = {
#     "gemini"    : {"nome": "Gemini 2.5 Flash",          "provedor": "google",      "modelo": "gemini-2.5-flash"},
#     "groq_llama": {"nome": "Llama 3.3 70B (Groq)",      "provedor": "groq",        "modelo": "llama-3.3-70b-versatile"},
#     "gemma"     : {"nome": "Gemma 3 12B (OpenRouter)",  "provedor": "openrouter",  "modelo": "google/gemma-3-12b-it:free"},
#     "qwen"      : {"nome": "Qwen 2.5 72B (OpenRouter)", "provedor": "openrouter",  "modelo": "qwen/qwen-2.5-72b-instruct"},
# }

# # FIX-B: instrução explícita contra citação do título entre aspas
# SYSTEM_PROMPT = """Você é o Signal — um serviço que transforma movimentos em mercados preditivos em notícias econômicas claras e envolventes.

# CONTEXTO DO PRODUTO:
# Mercados preditivos são plataformas onde pessoas investem dinheiro apostando na probabilidade de eventos futuros. O preço de cada contrato reflete o que milhares de investidores, coletivamente, acreditam que vai acontecer — como uma pesquisa de opinião com dinheiro em jogo. Quando esse preço muda de forma abrupta, é sinal de que algo relevante aconteceu no cenário econômico mundial.

# SEU PAPEL:
# Escrever uma notícia curta, envolvente e acessível sobre esse movimento — como um bom repórter econômico que sabe explicar finanças para qualquer pessoa, sem jargão desnecessário.

# ESTRUTURA OBRIGATÓRIA — 3 parágrafos em HTML (<p>...</p><p>...</p><p>...</p>):

# PARÁGRAFO 1 — O FATO (lide da notícia):
# • Comece com o que mudou. Use um verbo de ação. NUNCA comece com "O Signal detectou" ou similares.
# • INTEGRE o tema do contrato como oração subordinada — NUNCA como citação entre aspas.
#   ✓ CORRETO: "A probabilidade de o Fed cortar os juros na reunião de setembro caiu 14 pontos..."
#   ✓ CORRETO: "Investidores reduziram suas apostas em um corte de juros do Fed em setembro..."
#   ✗ ERRADO:  "As apostas sobre 'O Fed vai cortar os juros em setembro?' caíram..."
#   ✗ ERRADO:  "A probabilidade de que 'O Fed vai manter os juros' subiu..."
# • Cite OBRIGATORIAMENTE: (1) a probabilidade inicial, (2) a probabilidade final, (3) a variação em pontos percentuais.
# • Explique a intensidade SEM usar "z-score": use "uma variação [X] vezes fora do padrão histórico deste contrato".

# PARÁGRAFO 2 — O CONTEXTO:
# • Explique o que este contrato mede na prática e o que o movimento sugere.
# • Use linguagem condicional: "pode indicar", "sugere que", "aponta para".
# • Mencione o volume negociado (US$ X milhões/milhares) para dar dimensão.
# • Se resultado final conhecido: mencione se o mercado acertou ou errou.

# PARÁGRAFO 3 — O IMPACTO PARA O BRASIL (máximo 3 frases diretas):
# • VARIE o ângulo — não repita "dólar sobe, capital foge" em todo evento:
#   - Câmbio (real × dólar)
#   - Custo do crédito (financiamentos, hipotecas, dívida pública)
#   - Decisão do Copom (Banco Central do Brasil)
#   - Bolsa (Ibovespa, exportadoras, siderúrgicas)
#   - Custo de importações e inflação doméstica
#   - Para cripto: investidores brasileiros, exchanges, regulação CVM
# • Conecte o movimento específico ao impacto específico.
# • PROIBIDO começar com "Para o Brasil e outros mercados emergentes".

# REGRAS ABSOLUTAS:
# • Formato: EXATAMENTE <p>...</p><p>...</p><p>...</p>
# • Use <b>valor</b> apenas para números (probabilidades em %, variação em pp, volume em US$).
# • PROIBIDO: markdown (**texto**), bullet points, listas numeradas.
# • PROIBIDO: "z-score" pelo nome — diga "X vezes fora do padrão histórico".
# • PROIBIDO: inventar dados ou causas não fornecidas.
# • PROIBIDO: citar o título do contrato entre aspas no corpo do texto.

# SEMÂNTICA DOS CAMPOS:
# • prob_antes  = 3 dias ANTES do sinal (ponto de partida)
# • prob_yes    = NA DATA do sinal (pico ou vale)
# • prob_depois = 3 dias DEPOIS (confirmação ou reversão)
# • Sequência: prob_antes → prob_yes → prob_depois
# """


# def _montar_prompt(row):
#     zscore_abs    = abs(row["zscore"])
#     intensidade   = f"cerca de {zscore_abs:.0f} vezes fora do padrão histórico deste contrato"
#     direcao_verbo = "subiu" if row["direcao"] == "alta" else "caiu"
#     variacao_abs  = abs(row["variacao"])

#     if row["resultado"] == "Yes":
#         resultado_txt = "Sim — o mercado acertou a aposta"
#     elif row["resultado"] == "No":
#         resultado_txt = "Não — o mercado errou a aposta"
#     else:
#         resultado_txt = "ainda em aberto"

#     # FIX-C: variação destacada como campo obrigatório + instrução explícita
#     return f"""
# Dados do sinal:

# Categoria: {row.get('categoria', 'FOMC')}
# Tema do contrato (integre como oração, NÃO cite entre aspas): {row['questao_pt']}
# Contrato original em inglês (só referência): {row['questao']}
# Data do sinal: {row['data'].date()}

# OBRIGATÓRIO citar no Parágrafo 1:
#   • Probabilidade antes: {row['prob_antes']:.1f}%
#   • Probabilidade no sinal: {row['prob_yes']:.1f}%
#   • Variação: {direcao_verbo} {variacao_abs:.1f} pontos percentuais  ← CITE ESTE NÚMERO EXPLICITAMENTE
#   • Intensidade: {intensidade}

# Sequência completa para o Parágrafo 2:
#   {row['prob_antes']:.1f}% → {row['prob_yes']:.1f}% → {row['prob_depois']:.1f}% (3 dias depois)

# Volume total do contrato: US$ {row['volume']:,.0f}
# Resultado final: {resultado_txt}

# Escreva a notícia seguindo a estrutura do sistema.
# """


# def chamar_gemini(prompt):
#     r = client.models.generate_content(
#         model=MODELOS["gemini"]["modelo"],
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             system_instruction=SYSTEM_PROMPT,
#             temperature=0.4,
#             safety_settings=[
#                 types.SafetySetting(category=c, threshold=types.HarmBlockThreshold.BLOCK_NONE)
#                 for c in [
#                     types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
#                     types.HarmCategory.HARM_CATEGORY_HARASSMENT,
#                     types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
#                     types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
#                 ]
#             ],
#         ),
#     )
#     if r.candidates and r.candidates[0].finish_reason != types.FinishReason.STOP:
#         raise Exception(f"Geração interrompida: {r.candidates[0].finish_reason}")
#     if not r.text:
#         raise Exception("Resposta vazia do Gemini")
#     return r.text.strip()


# def chamar_groq(prompt):
#     resp = requests.post(
#         "https://api.groq.com/openai/v1/chat/completions",
#         headers={"Authorization": f"Bearer {GROQ_KEY}", "Content-Type": "application/json"},
#         json={
#             "model": MODELOS["groq_llama"]["modelo"],
#             "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
#             "max_tokens": 900, "temperature": 0.4,
#         },
#         timeout=40,
#     )
#     data = resp.json()
#     if resp.status_code == 429:
#         raise Exception(f"Groq rate limit: {data.get('error',{}).get('message','')}")
#     if "choices" not in data or not data["choices"]:
#         raise Exception(f"Groq sem choices: {data.get('error', data)}")
#     conteudo = data["choices"][0]["message"].get("content", "").strip()
#     if not conteudo:
#         raise Exception("Resposta vazia do Groq")
#     return conteudo


# def chamar_openrouter(prompt, modelo_id):
#     cfg  = MODELOS[modelo_id]
#     resp = requests.post(
#         "https://openrouter.ai/api/v1/chat/completions",
#         headers={
#             "Authorization": f"Bearer {OPENROUTER_KEY}",
#             "Content-Type": "application/json",
#             "HTTP-Referer": "https://signal-tcc.dev",
#             "X-Title": "Signal TCC",
#         },
#         json={
#             "model": cfg["modelo"],
#             "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}],
#             "max_tokens": 900, "temperature": 0.4,
#         },
#         timeout=40,
#     )
#     data = resp.json()
#     if "choices" not in data or not data["choices"]:
#         raise Exception(f"OpenRouter sem choices: {data.get('error', data)}")
#     conteudo = data["choices"][0]["message"].get("content", "").strip()
#     if not conteudo:
#         raise Exception("Resposta vazia do OpenRouter")
#     return conteudo


# def normalizar_narrativa(texto: str) -> str:
#     if not texto or texto.startswith("ERRO"):
#         return texto
#     texto = re.sub(r'\*\*(.*?)\*\*', r'<b>\1</b>', texto)
#     texto = re.sub(r'\*(.*?)\*',     r'<b>\1</b>', texto)
#     if "<p>" in texto or "</p>" in texto:
#         texto = re.sub(r'</p>\s*</p>', '</p>', texto)
#         texto = re.sub(r'<p>\s*<p>',   '<p>',  texto)
#     else:
#         texto = re.sub(r'\n{2,}', '</p><p>', texto)
#     if not texto.startswith("<p>"):
#         texto = "<p>" + texto
#     if not texto.endswith("</p>"):
#         texto = texto + "</p>"
#     texto = re.sub(r'<p>\s*</p>', '', texto)
#     return texto.strip()


# def waterfall(prompt):
#     ordem = [
#         ("gemini",     lambda p: chamar_gemini(p)),
#         ("groq_llama", lambda p: chamar_groq(p)),
#         ("gemma",      lambda p: chamar_openrouter(p, "gemma")),
#         ("qwen",       lambda p: chamar_openrouter(p, "qwen")),
#     ]
#     tentativas = []
#     for modelo_id, fn in ordem:
#         nome = MODELOS[modelo_id]["nome"]
#         try:
#             narrativa = fn(prompt)
#             tentativas.append({"modelo": nome, "status": "ok"})
#             return narrativa, nome, tentativas
#         except Exception as e:
#             tentativas.append({"modelo": nome, "status": f"erro: {e}"})
#             print(f"  [{nome}] falhou → tentando próximo...")
#             time.sleep(1)
#     return "ERRO: todos os modelos falharam.", "nenhum", tentativas


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 5 — Sistema de pontuação das narrativas
# # ═══════════════════════════════════════════════════════════════════

# def pontuar_narrativa(narrativa: str, row) -> tuple:
#     if not narrativa or narrativa.startswith("ERRO"):
#         return 0, {"erro": narrativa}

#     texto          = narrativa.lower()
#     texto_original = narrativa
#     scores         = {}

#     def num_presente(valor, txt):
#         candidatos = set()
#         try:
#             v = float(valor)
#             for fmt in [str(int(v)), str(round(v,1)), f"{v:.1f}", f"{v:.0f}", str(v).replace(".0","")]:
#                 candidatos.add(fmt)
#         except (ValueError, TypeError):
#             candidatos.add(str(valor))
#         return any(c in txt for c in candidatos)

#     p = 0
#     if num_presente(row["prob_yes"], texto_original):      p += 10
#     if num_presente(abs(row["variacao"]), texto_original): p += 10
#     if str(row["direcao"]).lower() in texto:               p += 5
#     if str(row["magnitude"]).lower() in texto:             p += 5
#     scores["ancoragem_factual"] = p

#     n_p = len(re.findall(r'<p>', texto_original))
#     if n_p == 0:
#         n_p = max(1, len(texto_original.split("\n\n")))
#     scores["extensao"] = 25 if 2 <= n_p <= 3 else (15 if n_p in [1, 4] else 5)

#     termos = [
#         "federal reserve", "fed", "juros", "taxa", "política monetária",
#         "mercado", "probabilidade", "investidor", "economia", "corte",
#         "inflação", "recessão", "brasil", "câmbio", "bps", "fomc",
#         "emergente", "dólar", "monetary", "interest", "copom", "ibovespa",
#         "real", "crédito", "banco central", "bitcoin", "cripto", "cpi",
#         "pib", "desemprego", "tarifa",
#     ]
#     scores["contexto_economico"] = min(25, sum(1 for t in termos if t in texto) * 3)

#     jargao = ["z-score", "brier score", "standard deviation",
#               "changepoint", "zscore", "fidelity", "sliding window"]
#     abertura_proib = texto_original.lower().lstrip("<p>").startswith(
#         ("o signal detectou", "um movimento foi detectado", "uma anomalia foi")
#     )
#     # FIX-B: penaliza título entre aspas no corpo
#     titulo_entre_aspas = bool(re.search(r'["""«»]O Fed|["""«»]A taxa|["""«»]Os EUA|["""«»]O Bitcoin', texto_original))
#     penalidade = (sum(4 for j in jargao if j in texto)
#                   + (10 if abertura_proib else 0)
#                   + (5 if titulo_entre_aspas else 0))
#     scores["clareza"] = max(0, 20 - penalidade)

#     return sum(scores.values()), scores


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 6 — Pipeline principal
# # ═══════════════════════════════════════════════════════════════════

# def calcular_brier_score(eventos: list):
#     validos = [e for e in eventos if e.get("resultado") in ["Yes", "No"]]
#     if not validos:
#         return None
#     soma = sum(((e["prob_yes"]/100) - (1 if e["resultado"]=="Yes" else 0))**2 for e in validos)
#     return round(soma / len(validos), 4)


# def rodar_pipeline(mercados_coletados, limiar_z=None, max_novos=20, df_sensibilidade=None):
#     limiar_z = limiar_z or PARAMS["limiar_z_padrao"]

#     print(f"\nDetectando signals com z* = {limiar_z}...")
#     todos = [s for m in mercados_coletados
#              for s in [detectar_signals(m, limiar_z=limiar_z)] if not s.empty]
#     if not todos:
#         print("Nenhum signal detectado.")
#         return []

#     df_dedup = deduplicar_signals(pd.concat(todos, ignore_index=True))
#     print(f"Signals únicos: {len(df_dedup)}")

#     # Mostra distribuição de magnitude para validação
#     mags = df_dedup["magnitude"].value_counts().to_dict()
#     print(f"  Distribuição de magnitude: {mags}")
#     cats = df_dedup.get("categoria", pd.Series()).value_counts().to_dict() if "categoria" in df_dedup else {}
#     if cats:
#         print(f"  Por categoria: {cats}")

#     processados = carregar_processados()
#     df_novos = df_dedup[
#         ~df_dedup.apply(lambda r: signal_id(r) in processados, axis=1)
#     ].head(max_novos)
#     print(f"Novos para processar: {len(df_novos)} (já processados: {len(df_dedup)-len(df_novos)})")

#     if df_novos.empty:
#         print("Feed já atualizado.")
#         return _carregar_feed()

#     feed = _carregar_feed()

#     for idx, (_, row) in enumerate(df_novos.iterrows(), 1):
#         sid        = signal_id(row)
#         questao_pt = traduzir_questao(row["questao"])
#         print(f"\n[{idx}/{len(df_novos)}] {row['data'].date()} z={row['zscore']:.2f} [{row.get('categoria','?')}] {row['questao'][:50]}")
#         print(f"  PT: {questao_pt}")

#         row_com_pt           = row.copy()
#         row_com_pt["questao_pt"] = questao_pt

#         prompt    = _montar_prompt(row_com_pt)
#         narrativa, modelo_usado, tentativas = waterfall(prompt)
#         narrativa = normalizar_narrativa(narrativa)
#         score, detalhes = pontuar_narrativa(narrativa, row)

#         z_abs = abs(float(row["zscore"]))
#         entrada = {
#             "id"                  : sid,
#             "data"                : str(row["data"].date()),
#             "questao"             : row["questao"],
#             "questao_pt"          : questao_pt,
#             "evento"              : row["evento"],
#             "categoria"           : row.get("categoria", "FOMC"),
#             "prob_yes"            : round(float(row["prob_yes"]), 1),
#             "variacao"            : round(float(row["variacao"]), 2),
#             "zscore"              : round(float(row["zscore"]), 2),
#             "magnitude"           : row["magnitude"],
#             "magnitude_descricao" : magnitude_descricao(row["magnitude"], z_abs),  # FIX-A
#             "direcao"             : row["direcao"],
#             "prob_antes"          : round(float(row["prob_antes"]), 1),
#             "prob_depois"         : round(float(row["prob_depois"]), 1),
#             "volume"              : float(row["volume"]),
#             "end_date"            : row["end_date"],
#             "resultado"           : row["resultado"],
#             "narrativa"           : narrativa,
#             "modelo_usado"        : modelo_usado,
#             "score"               : score,
#             "score_detalhes"      : detalhes,
#             "tentativas"          : tentativas,
#             "token_id"            : row["token_id"],
#             "gerado_em"           : datetime.now(timezone.utc).isoformat(),
#         }

#         feed.append(entrada)
#         processados.add(sid)
#         salvar_processados(processados)
#         print(f"  Modelo: {modelo_usado} | Score: {score}/100")
#         time.sleep(3)

#     feed.sort(key=lambda x: x["data"], reverse=True)
#     brier = calcular_brier_score(feed)
#     _salvar_feed(feed, brier_score=brier, df_sensibilidade=df_sensibilidade)

#     print(f"\n✓ Feed atualizado: {len(feed)} eventos totais")
#     if brier is not None:
#         n_val = sum(1 for e in feed if e.get("resultado") in ["Yes","No"])
#         print(f"  Brier Score: {brier:.4f} (sobre {n_val} eventos com desfecho)")
#     return feed


# def _carregar_feed():
#     if os.path.exists(FEED_FILE):
#         with open(FEED_FILE, "r", encoding="utf-8") as f:
#             raw = json.load(f)
#         eventos = raw.get("events", [])
#         vistos = {}
#         for e in eventos:
#             eid = e.get("id","")
#             if eid not in vistos or e.get("gerado_em","") > vistos[eid].get("gerado_em",""):
#                 vistos[eid] = e
#         return list(vistos.values())
#     return []


# def _salvar_feed(eventos, brier_score=None, df_sensibilidade=None):
#     vistos = {}
#     for e in eventos:
#         eid = e.get("id","")
#         if eid not in vistos or e.get("gerado_em","") > vistos[eid].get("gerado_em",""):
#             vistos[eid] = e
#     eventos_unicos = sorted(vistos.values(), key=lambda x: x["data"], reverse=True)
#     sens_list = df_sensibilidade.to_dict(orient="records") if df_sensibilidade is not None and not df_sensibilidade.empty else None
#     payload = {
#         "gerado_em"    : datetime.now(timezone.utc).isoformat(),
#         "threshold"    : PARAMS["limiar_z_padrao"],
#         "janela"       : PARAMS["janela_zscore"],
#         "brier_score"  : brier_score,
#         "sensibilidade": sens_list,
#         "events"       : eventos_unicos,
#     }
#     with open(FEED_FILE, "w", encoding="utf-8") as f:
#         json.dump(payload, f, ensure_ascii=False, indent=2, default=str)


# def exportar_serie_para_frontend(mercados_coletados, token_id_alvo: str):
#     for m in mercados_coletados:
#         if m["token_id"] != token_id_alvo:
#             continue
#         df = m["df"].copy()
#         df["variacao"] = df["prob_yes"].diff()
#         vol = (df["variacao"].rolling(PARAMS["janela_zscore"], min_periods=5)
#                              .std().bfill().clip(lower=PARAMS["vol_piso"]))
#         df["zscore"] = (df["variacao"] / vol).round(2)
#         df["dia"]    = df["data"].dt.strftime("%d/%m")
#         df_diario    = df.groupby("dia", sort=False).last().reset_index()
#         if not os.path.exists(FEED_FILE):
#             print(f"ERRO: {FEED_FILE} não encontrado.")
#             return
#         with open(FEED_FILE, "r", encoding="utf-8") as f:
#             payload = json.load(f)
#         payload["serie"] = {
#             "token_id"  : token_id_alvo,
#             "questao"   : m["questao"],
#             "questao_pt": traduzir_questao(m["questao"]),
#             "dates"     : df_diario["dia"].tolist(),
#             "probs"     : df_diario["prob_yes"].round(1).tolist(),
#             "zscores"   : df_diario["zscore"].fillna(0).tolist(),
#         }
#         with open(FEED_FILE, "w", encoding="utf-8") as f:
#             json.dump(payload, f, ensure_ascii=False, indent=2, default=str)
#         print(f"✓ Série exportada: {len(df_diario)} pontos → {FEED_FILE}")
#         print(f"  Contrato PT: {traduzir_questao(m['questao'])}")
#         return
#     print(f"ERRO: token_id '{token_id_alvo}' não encontrado.")


# # ═══════════════════════════════════════════════════════════════════
# # CÉLULA 7 — Execução
# # ═══════════════════════════════════════════════════════════════════

# if __name__ == "__main__":

#     # Descomente se precisar reprocessar IDs específicos:
#     # forcar_reprocessamento({"id1", "id2"})

#     print("=== PASSO 1: Coleta de dados ===")
#     mercados_coletados = coletar_mercados_fomc()

#     print("\n=== PASSO 2: Análise de sensibilidade ===")
#     df_sensibilidade = analise_sensibilidade(mercados_coletados)
#     df_sensibilidade.to_csv("sensibilidade.csv", index=False)
#     print("  → Salvo em sensibilidade.csv")

#     print("\n=== PASSO 3: Pipeline de narrativas ===")
#     feed = rodar_pipeline(
#         mercados_coletados,
#         limiar_z=2.0,
#         max_novos=15,
#         df_sensibilidade=df_sensibilidade,
#     )

#     if feed:
#         print("\n=== PASSO 4: Exportar série temporal ===")
#         tokens_mercados = {m["token_id"]: m for m in mercados_coletados}
#         alvo = max(
#             (ev for ev in feed if ev.get("token_id") in tokens_mercados),
#             key=lambda ev: tokens_mercados[ev["token_id"]]["volume"],
#             default=None,
#         )
#         if alvo:
#             exportar_serie_para_frontend(mercados_coletados, alvo["token_id"])

#     print(f"\n✓ Concluído. Arquivo: {FEED_FILE}")
#     with open(FEED_FILE, "r") as f:
#         check = json.load(f)
#     cats = {}
#     mags = {}
#     for e in check.get("events", []):
#         c = e.get("categoria","?"); cats[c] = cats.get(c,0)+1
#         mg = e.get("magnitude","?"); mags[mg] = mags.get(mg,0)+1
#     print(f"\nVerificação do {FEED_FILE}:")
#     print(f"  Eventos      : {len(check.get('events',[]))}")
#     print(f"  Por categoria: {cats}")
#     print(f"  Magnitudes   : {mags}")
#     print(f"  Brier Score  : {check.get('brier_score','—')}")
#     print(f"  Sensibilidade: {'sim (' + str(len(check['sensibilidade'])) + ' thresholds)' if check.get('sensibilidade') else 'não'}")
#     print(f"  Serie        : {'sim (' + str(len(check['serie']['dates'])) + ' pontos)' if check.get('serie') else 'não'}")
#     print(f"  Threshold    : {check.get('threshold')}")
#     print(f"  Gerado em    : {check.get('gerado_em')}")
